In [1]:
# ============================================================
# 03.06 GAN TRAINING
# CELL 1 — IMPORTS AND ENVIRONMENT VERIFICATION
# ============================================================

from pathlib import Path
import json
import copy
import random

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


print("=" * 100)
print("03.06 GAN TRAINING — ENVIRONMENT VERIFICATION")
print("=" * 100)

print("\nPython / numerical libraries loaded successfully.")

print("PyTorch version :", torch.__version__)
print("NumPy version   :", np.__version__)

print("\nCUDA available  :", torch.cuda.is_available())

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Training device :", device)

print("\n✓ REQUIRED LIBRARIES IMPORTED")

03.06 GAN TRAINING — ENVIRONMENT VERIFICATION

Python / numerical libraries loaded successfully.
PyTorch version : 2.13.0+cpu
NumPy version   : 2.5.2

CUDA available  : False
Training device : cpu

✓ REQUIRED LIBRARIES IMPORTED


In [2]:
# ============================================================
# CELL 2 — REPRODUCIBILITY CONFIGURATION
# ============================================================

SEED = 42


# ------------------------------------------------------------
# Python random seed
# ------------------------------------------------------------

random.seed(SEED)


# ------------------------------------------------------------
# NumPy random seed
# ------------------------------------------------------------

np.random.seed(SEED)


# ------------------------------------------------------------
# PyTorch random seed
# ------------------------------------------------------------

torch.manual_seed(SEED)


# ------------------------------------------------------------
# CUDA seed
# ------------------------------------------------------------

if torch.cuda.is_available():

    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)


# ------------------------------------------------------------
# Deterministic backend configuration
# ------------------------------------------------------------

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


print("=" * 100)
print("03.06 GAN TRAINING — REPRODUCIBILITY CONFIGURATION")
print("=" * 100)

print("Random seed             :", SEED)
print("PyTorch seed configured :", True)
print("NumPy seed configured   :", True)
print("Python seed configured :", True)

print(
    "CUDA seed configured   :",
    torch.cuda.is_available()
)

print(
    "CuDNN deterministic    :",
    torch.backends.cudnn.deterministic
)

print(
    "CuDNN benchmark        :",
    torch.backends.cudnn.benchmark
)

print("\n✓ REPRODUCIBILITY CONFIGURATION COMPLETE")

03.06 GAN TRAINING — REPRODUCIBILITY CONFIGURATION
Random seed             : 42
PyTorch seed configured : True
NumPy seed configured   : True
Python seed configured : True
CUDA seed configured   : False
CuDNN deterministic    : True
CuDNN benchmark        : False

✓ REPRODUCIBILITY CONFIGURATION COMPLETE


In [3]:
# ============================================================
# CELL 3 — ROBUST PROJECT ROOT DISCOVERY
# ============================================================

current_path = Path.cwd()

project_root = None

for candidate in [current_path] + list(current_path.parents):

    phase_root_candidate = (
        candidate / "06_cross_modal_learning"
    )

    if phase_root_candidate.exists():

        project_root = candidate
        break


if project_root is None:

    raise FileNotFoundError(
        "Could not locate the project root.\n"
        "Expected a folder named '06_cross_modal_learning' "
        "somewhere above the notebook."
    )


phase_root = (
    project_root / "06_cross_modal_learning"
)

official_phase3_root = (
    phase_root / "03_cross_modal_learning"
)


print("=" * 100)
print("03.06 GAN TRAINING — PROJECT PATH VERIFICATION")
print("=" * 100)

print("\nCurrent notebook working directory:")
print(current_path)

print("\nProject root:")
print(project_root)

print("\n06_cross_modal_learning:")
print(phase_root)

print("\nOfficial Phase 03:")
print(official_phase3_root)


assert phase_root.exists()
assert official_phase3_root.exists()


print("\n✓ PROJECT ROOT FOUND")
print("✓ OFFICIAL PHASE 03 ROOT FOUND")

03.06 GAN TRAINING — PROJECT PATH VERIFICATION

Current notebook working directory:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\03_cross_modal_learning\03_06_gan_training

Project root:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening

06_cross_modal_learning:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning

Official Phase 03:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\03_cross_modal_learning

✓ PROJECT ROOT FOUND
✓ OFFICIAL PHASE 03 ROOT FOUND


In [4]:
# ============================================================
# CELL 4 — LOAD APPROVED PHASE 02.08 PAIR DATASET
# ============================================================

pair_dataset_dir = (
    phase_root
    / "02_cross_modal_pairing"
    / "pairs"
    / "representation_aware_split"
)


train_pairs_path = (
    pair_dataset_dir
    / "02_08_train_representation_disjoint_pairs.npz"
)

validation_pairs_path = (
    pair_dataset_dir
    / "02_08_validation_representation_disjoint_pairs.npz"
)


print("=" * 100)
print("03.06 GAN TRAINING — PHASE 02.08 DATASET LOADING")
print("=" * 100)

print("\nPair dataset directory:")
print(pair_dataset_dir)

print("\nTraining pair file:")
print(train_pairs_path)

print("\nValidation pair file:")
print(validation_pairs_path)


# ------------------------------------------------------------
# Safety checks BEFORE loading
# ------------------------------------------------------------

if not pair_dataset_dir.exists():

    raise FileNotFoundError(
        f"Pair dataset directory not found:\n"
        f"{pair_dataset_dir}"
    )


if not train_pairs_path.exists():

    raise FileNotFoundError(
        f"Training pair file not found:\n"
        f"{train_pairs_path}"
    )


if not validation_pairs_path.exists():

    raise FileNotFoundError(
        f"Validation pair file not found:\n"
        f"{validation_pairs_path}"
    )


# ------------------------------------------------------------
# Load NPZ files
# ------------------------------------------------------------

train_pairs = np.load(
    train_pairs_path
)

validation_pairs = np.load(
    validation_pairs_path
)


# ------------------------------------------------------------
# Extract arrays
# ------------------------------------------------------------

train_source = train_pairs["source_embeddings"]
train_target = train_pairs["target_embeddings"]
train_labels = train_pairs["pair_labels"]


validation_source = (
    validation_pairs["source_embeddings"]
)

validation_target = (
    validation_pairs["target_embeddings"]
)

validation_labels = (
    validation_pairs["pair_labels"]
)


print("\n" + "-" * 100)
print("DATASET LOADED")
print("-" * 100)

print(
    "Training source shape     :",
    train_source.shape
)

print(
    "Training target shape     :",
    train_target.shape
)

print(
    "Training labels shape     :",
    train_labels.shape
)

print(
    "Validation source shape   :",
    validation_source.shape
)

print(
    "Validation target shape   :",
    validation_target.shape
)

print(
    "Validation labels shape   :",
    validation_labels.shape
)


print("\n✓ PHASE 02.08 DATASET LOADED SUCCESSFULLY")

03.06 GAN TRAINING — PHASE 02.08 DATASET LOADING

Pair dataset directory:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\02_cross_modal_pairing\pairs\representation_aware_split

Training pair file:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\02_cross_modal_pairing\pairs\representation_aware_split\02_08_train_representation_disjoint_pairs.npz

Validation pair file:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\02_cross_modal_pairing\pairs\representation_aware_split\02_08_validation_representation_disjoint_pairs.npz

----------------------------------------------------------------------------------------------------
DATASET LOADED
----------------------------------------------------------------------------------------------------
Training source shape     : (1776, 32)
Training target shape     : (1776, 32)
Training labels shape     : (1776,)
Val

In [5]:
# ============================================================
# CELL 5 — PHASE 02.08 DATASET INTEGRITY AUDIT
# ============================================================

print("=" * 100)
print("03.06 GAN TRAINING — PHASE 02.08 DATASET INTEGRITY AUDIT")
print("=" * 100)


# ------------------------------------------------------------
# Expected dimensions
# ------------------------------------------------------------

EXPECTED_TRAIN_SAMPLES = 1776
EXPECTED_VALIDATION_SAMPLES = 448
EXPECTED_EMBEDDING_DIM = 32


# ------------------------------------------------------------
# Shape checks
# ------------------------------------------------------------

assert train_source.shape == (
    EXPECTED_TRAIN_SAMPLES,
    EXPECTED_EMBEDDING_DIM
)

assert train_target.shape == (
    EXPECTED_TRAIN_SAMPLES,
    EXPECTED_EMBEDDING_DIM
)

assert train_labels.shape == (
    EXPECTED_TRAIN_SAMPLES,
)


assert validation_source.shape == (
    EXPECTED_VALIDATION_SAMPLES,
    EXPECTED_EMBEDDING_DIM
)

assert validation_target.shape == (
    EXPECTED_VALIDATION_SAMPLES,
    EXPECTED_EMBEDDING_DIM
)

assert validation_labels.shape == (
    EXPECTED_VALIDATION_SAMPLES,
)


# ------------------------------------------------------------
# Convert labels to integer arrays for auditing
# ------------------------------------------------------------

train_labels_int = train_labels.astype(
    np.int64
)

validation_labels_int = (
    validation_labels.astype(np.int64)
)


# ------------------------------------------------------------
# Label validity
# ------------------------------------------------------------

train_unique_labels = np.unique(
    train_labels_int
)

validation_unique_labels = np.unique(
    validation_labels_int
)


assert set(
    train_unique_labels.tolist()
).issubset({0, 1})


assert set(
    validation_unique_labels.tolist()
).issubset({0, 1})


# ------------------------------------------------------------
# Class counts
# ------------------------------------------------------------

train_positive = int(
    np.sum(train_labels_int == 1)
)

train_negative = int(
    np.sum(train_labels_int == 0)
)

validation_positive = int(
    np.sum(validation_labels_int == 1)
)

validation_negative = int(
    np.sum(validation_labels_int == 0)
)


# ------------------------------------------------------------
# Finite-value checks
# ------------------------------------------------------------

train_source_finite = np.isfinite(
    train_source
).all()

train_target_finite = np.isfinite(
    train_target
).all()

validation_source_finite = np.isfinite(
    validation_source
).all()

validation_target_finite = np.isfinite(
    validation_target
).all()


assert train_source_finite
assert train_target_finite
assert validation_source_finite
assert validation_target_finite


# ------------------------------------------------------------
# Print audit
# ------------------------------------------------------------

print("\nTRAINING DATA")
print("-" * 60)

print(
    "Training samples       :",
    len(train_labels)
)

print(
    "Training positive      :",
    train_positive
)

print(
    "Training negative      :",
    train_negative
)

print(
    "Source dimension       :",
    train_source.shape[1]
)

print(
    "Target dimension       :",
    train_target.shape[1]
)


print("\nVALIDATION DATA")
print("-" * 60)

print(
    "Validation samples     :",
    len(validation_labels)
)

print(
    "Validation positive    :",
    validation_positive
)

print(
    "Validation negative    :",
    validation_negative
)

print(
    "Source dimension       :",
    validation_source.shape[1]
)

print(
    "Target dimension       :",
    validation_target.shape[1]
)


print("\nDATA QUALITY")
print("-" * 60)

print(
    "Training source finite     :",
    train_source_finite
)

print(
    "Training target finite     :",
    train_target_finite
)

print(
    "Validation source finite   :",
    validation_source_finite
)

print(
    "Validation target finite   :",
    validation_target_finite
)

print(
    "Training labels valid      :",
    set(train_unique_labels.tolist()) == {0, 1}
)

print(
    "Validation labels valid    :",
    set(validation_unique_labels.tolist()) == {0, 1}
)


# ------------------------------------------------------------
# Final assertions
# ------------------------------------------------------------

assert train_positive == 888
assert train_negative == 888

assert validation_positive == 224
assert validation_negative == 224


print("\n" + "=" * 100)
print("✓ PHASE 02.08 DATASET INTEGRITY AUDIT PASSED")
print("=" * 100)

print("""
The GAN training dataset is:

Training   : 1,776 pairs
Validation :   448 pairs

Training balance:
    Positive = 888
    Negative = 888

Validation balance:
    Positive = 224
    Negative = 224

Embedding dimension:
    Source = 32-D
    Target = 32-D

NaN / Inf:
    None detected

Label validity:
    PASSED
""")

03.06 GAN TRAINING — PHASE 02.08 DATASET INTEGRITY AUDIT

TRAINING DATA
------------------------------------------------------------
Training samples       : 1776
Training positive      : 888
Training negative      : 888
Source dimension       : 32
Target dimension       : 32

VALIDATION DATA
------------------------------------------------------------
Validation samples     : 448
Validation positive    : 224
Validation negative    : 224
Source dimension       : 32
Target dimension       : 32

DATA QUALITY
------------------------------------------------------------
Training source finite     : True
Training target finite     : True
Validation source finite   : True
Validation target finite   : True
Training labels valid      : True
Validation labels valid    : True

✓ PHASE 02.08 DATASET INTEGRITY AUDIT PASSED

The GAN training dataset is:

Training   : 1,776 pairs
Validation :   448 pairs

Training balance:
    Positive = 888
    Negative = 888

Validation balance:
    Positive = 224

In [6]:
# ============================================================
# CELL 6 — GENERATOR DEFINITION
# ============================================================

class Generator(nn.Module):
    """
    Generator for unpaired cross-modal representation learning.

    Input:
        Source modality representation: 32-D

    Output:
        Generated target-like representation: 32-D

    Architecture:
        32 -> 64 -> 128 -> 64 -> 32
    """

    def __init__(self, input_dim=32, output_dim=32):

        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(input_dim, 64),
            nn.ReLU(),

            nn.Linear(64, 128),
            nn.ReLU(),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, output_dim)
        )


    def forward(self, source):

        return self.network(source)


# ------------------------------------------------------------
# Instantiate Generator
# ------------------------------------------------------------

generator = Generator(
    input_dim=32,
    output_dim=32
).to(device)


print("=" * 100)
print("03.06 GAN TRAINING — GENERATOR")
print("=" * 100)

print("\nGenerator architecture:\n")
print(generator)


# ------------------------------------------------------------
# Parameter count
# ------------------------------------------------------------

generator_total_parameters = sum(
    p.numel()
    for p in generator.parameters()
)

generator_trainable_parameters = sum(
    p.numel()
    for p in generator.parameters()
    if p.requires_grad
)


print("\nTotal Generator parameters     :",
      generator_total_parameters)

print("Trainable Generator parameters :",
      generator_trainable_parameters)

print("\nInput dimension  : 32")
print("Output dimension : 32")

print("\n✓ GENERATOR DEFINED")

03.06 GAN TRAINING — GENERATOR

Generator architecture:

Generator(
  (network): Sequential(
    (0): Linear(in_features=32, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=128, bias=True)
    (3): ReLU()
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): ReLU()
    (6): Linear(in_features=64, out_features=32, bias=True)
  )
)

Total Generator parameters     : 20768
Trainable Generator parameters : 20768

Input dimension  : 32
Output dimension : 32

✓ GENERATOR DEFINED


In [7]:
# ============================================================
# CELL 7 — DISCRIMINATOR DEFINITION
# ============================================================

class Discriminator(nn.Module):
    """
    Discriminator for real-vs-generated target representations.

    Input:
        Target representation: 32-D

    Output:
        Single real/fake logit

    Architecture:
        32 -> 64 -> 32 -> 1
    """

    def __init__(self, input_dim=32):

        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(input_dim, 64),
            nn.ReLU(),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 1)
        )


    def forward(self, target_representation):

        return self.network(
            target_representation
        ).squeeze(-1)


# ------------------------------------------------------------
# Instantiate Discriminator
# ------------------------------------------------------------

discriminator = Discriminator(
    input_dim=32
).to(device)


print("=" * 100)
print("03.06 GAN TRAINING — DISCRIMINATOR")
print("=" * 100)

print("\nDiscriminator architecture:\n")
print(discriminator)


# ------------------------------------------------------------
# Parameter count
# ------------------------------------------------------------

discriminator_total_parameters = sum(
    p.numel()
    for p in discriminator.parameters()
)

discriminator_trainable_parameters = sum(
    p.numel()
    for p in discriminator.parameters()
    if p.requires_grad
)


print("\nTotal Discriminator parameters     :",
      discriminator_total_parameters)

print(
    "Trainable Discriminator parameters :",
    discriminator_trainable_parameters
)

print("\nInput dimension  : 32")
print("Output dimension : 1 logit")

print("\n✓ DISCRIMINATOR DEFINED")

03.06 GAN TRAINING — DISCRIMINATOR

Discriminator architecture:

Discriminator(
  (network): Sequential(
    (0): Linear(in_features=32, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=32, bias=True)
    (3): ReLU()
    (4): Linear(in_features=32, out_features=1, bias=True)
  )
)

Total Discriminator parameters     : 4225
Trainable Discriminator parameters : 4225

Input dimension  : 32
Output dimension : 1 logit

✓ DISCRIMINATOR DEFINED


In [8]:
# ============================================================
# CELL 8 — LOAD FROZEN BASELINE RELATIONSHIP MODEL
# ============================================================

baseline_checkpoint_path = (
    official_phase3_root
    / "03_04_baseline_training"
    / "checkpoints"
    / "03_04_baseline_best_model.pt"
)


print("=" * 100)
print("03.06 GAN TRAINING — BASELINE CHECKPOINT VERIFICATION")
print("=" * 100)

print("\nExpected baseline checkpoint:")
print(baseline_checkpoint_path)


# ------------------------------------------------------------
# Safety check
# ------------------------------------------------------------

if not baseline_checkpoint_path.exists():

    raise FileNotFoundError(
        "OFFICIAL BASELINE CHECKPOINT NOT FOUND.\n\n"
        f"Expected path:\n{baseline_checkpoint_path}\n\n"
        "Training must not continue until this file exists."
    )


print("\n✓ Baseline checkpoint file exists.")


# ------------------------------------------------------------
# Load checkpoint
# ------------------------------------------------------------

baseline_checkpoint = torch.load(
    baseline_checkpoint_path,
    map_location=device
)


print("\nCheckpoint loaded successfully.")

print(
    "\nCheckpoint object type:",
    type(baseline_checkpoint)
)


# ------------------------------------------------------------
# Inspect checkpoint structure
# ------------------------------------------------------------

if isinstance(baseline_checkpoint, dict):

    print("\nCheckpoint dictionary keys:")

    for key in baseline_checkpoint.keys():

        print("  -", key)

else:

    print(
        "\nCheckpoint is not a dictionary."
    )


print("\n✓ BASELINE CHECKPOINT LOAD PASSED")

03.06 GAN TRAINING — BASELINE CHECKPOINT VERIFICATION

Expected baseline checkpoint:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\03_cross_modal_learning\03_04_baseline_training\checkpoints\03_04_baseline_best_model.pt

✓ Baseline checkpoint file exists.

Checkpoint loaded successfully.

Checkpoint object type: <class 'dict'>

Checkpoint dictionary keys:
  - epoch
  - model_state_dict
  - optimizer_state_dict
  - best_val_roc_auc
  - seed
  - batch_size
  - learning_rate
  - num_epochs

✓ BASELINE CHECKPOINT LOAD PASSED


In [9]:
# ============================================================
# CELL 9 — RECONSTRUCT AND LOAD FROZEN BASELINE
# ============================================================

class BaselineCrossModalModel(nn.Module):
    """
    Frozen semantic relationship model.

    Inputs:
        source : 32-D
        target : 32-D

    Relationship representation:
        [source,
         target,
         |source - target|,
         source * target]

    Total dimension:
        32 + 32 + 32 + 32 = 128

    Classifier:
        128 -> 128 -> 64 -> 1
    """

    def __init__(self):

        super().__init__()

        self.classifier = nn.Sequential(

            nn.Linear(128, 128),
            nn.ReLU(),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, 1)
        )


    def forward(self, source, target):

        relationship_features = torch.cat(
            [
                source,
                target,
                torch.abs(source - target),
                source * target
            ],
            dim=1
        )

        return self.classifier(
            relationship_features
        ).squeeze(-1)


# ------------------------------------------------------------
# Create model
# ------------------------------------------------------------

baseline_model = BaselineCrossModalModel().to(device)


# ------------------------------------------------------------
# Extract state dictionary
# ------------------------------------------------------------

if isinstance(baseline_checkpoint, dict):

    if "model_state_dict" in baseline_checkpoint:

        baseline_state_dict = (
            baseline_checkpoint["model_state_dict"]
        )

    elif "state_dict" in baseline_checkpoint:

        baseline_state_dict = (
            baseline_checkpoint["state_dict"]
        )

    else:

        # Check whether the dictionary itself looks like
        # a PyTorch state dictionary.
        tensor_values = all(
            isinstance(value, torch.Tensor)
            for value in baseline_checkpoint.values()
        )

        if tensor_values:

            baseline_state_dict = baseline_checkpoint

        else:

            raise KeyError(
                "Could not identify the model state dictionary "
                "inside the baseline checkpoint."
            )

else:

    baseline_state_dict = baseline_checkpoint


# ------------------------------------------------------------
# Load weights strictly
# ------------------------------------------------------------

baseline_model.load_state_dict(
    baseline_state_dict,
    strict=True
)


print("=" * 100)
print("03.06 GAN TRAINING — BASELINE MODEL LOADED")
print("=" * 100)

print("\nBaseline architecture:")
print(baseline_model)

print(
    "\nNumber of parameters:",
    sum(
        p.numel()
        for p in baseline_model.parameters()
    )
)

print(
    "\n✓ BASELINE STATE DICTIONARY LOADED WITH strict=True"
)

print(
    "✓ EXACT ARCHITECTURE MATCH CONFIRMED"
)

03.06 GAN TRAINING — BASELINE MODEL LOADED

Baseline architecture:
BaselineCrossModalModel(
  (classifier): Sequential(
    (0): Linear(in_features=128, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=1, bias=True)
  )
)

Number of parameters: 24833

✓ BASELINE STATE DICTIONARY LOADED WITH strict=True
✓ EXACT ARCHITECTURE MATCH CONFIRMED


In [10]:
# ============================================================
# CELL 10 — FREEZE AND VERIFY SEMANTIC BASELINE
# ============================================================

# ------------------------------------------------------------
# Freeze every baseline parameter
# ------------------------------------------------------------

for parameter in baseline_model.parameters():

    parameter.requires_grad = False


# ------------------------------------------------------------
# Evaluation mode
# ------------------------------------------------------------

baseline_model.eval()


# ------------------------------------------------------------
# Generator / Discriminator remain trainable
# ------------------------------------------------------------

generator.train()
discriminator.train()


# ------------------------------------------------------------
# Parameter audit
# ------------------------------------------------------------

baseline_total_parameters = sum(
    p.numel()
    for p in baseline_model.parameters()
)

baseline_trainable_parameters = sum(
    p.numel()
    for p in baseline_model.parameters()
    if p.requires_grad
)


generator_trainable_parameters = sum(
    p.numel()
    for p in generator.parameters()
    if p.requires_grad
)

discriminator_trainable_parameters = sum(
    p.numel()
    for p in discriminator.parameters()
    if p.requires_grad
)


print("=" * 100)
print("03.06 GAN TRAINING — MODEL FREEZING AUDIT")
print("=" * 100)

print("\nBASELINE SEMANTIC MODEL")
print("-" * 70)

print(
    "Total parameters     :",
    baseline_total_parameters
)

print(
    "Trainable parameters :",
    baseline_trainable_parameters
)

print(
    "Evaluation mode      :",
    not baseline_model.training
)


print("\nGENERATOR")
print("-" * 70)

print(
    "Trainable parameters :",
    generator_trainable_parameters
)

print(
    "Training mode        :",
    generator.training
)


print("\nDISCRIMINATOR")
print("-" * 70)

print(
    "Trainable parameters :",
    discriminator_trainable_parameters
)

print(
    "Training mode        :",
    discriminator.training
)


# ------------------------------------------------------------
# Safety assertions
# ------------------------------------------------------------

assert baseline_trainable_parameters == 0

assert generator_trainable_parameters > 0

assert discriminator_trainable_parameters > 0

assert baseline_model.training is False

assert generator.training is True

assert discriminator.training is True


print("\n" + "=" * 100)
print("✓ BASELINE SUCCESSFULLY FROZEN")
print("✓ GENERATOR IS TRAINABLE")
print("✓ DISCRIMINATOR IS TRAINABLE")
print("=" * 100)

03.06 GAN TRAINING — MODEL FREEZING AUDIT

BASELINE SEMANTIC MODEL
----------------------------------------------------------------------
Total parameters     : 24833
Trainable parameters : 0
Evaluation mode      : True

GENERATOR
----------------------------------------------------------------------
Trainable parameters : 20768
Training mode        : True

DISCRIMINATOR
----------------------------------------------------------------------
Trainable parameters : 4225
Training mode        : True

✓ BASELINE SUCCESSFULLY FROZEN
✓ GENERATOR IS TRAINABLE
✓ DISCRIMINATOR IS TRAINABLE


In [11]:
# ============================================================
# CELL 11 — GAN DATASET DEFINITION
# ============================================================

class CrossModalGANDataset(Dataset):
    """
    Dataset for GAN-based cross-modal representation learning.

    Each sample contains:
        source representation : 32-D
        real target           : 32-D
        relationship label    : 0 or 1
    """

    def __init__(self, source, target, labels):

        self.source = torch.tensor(
            source,
            dtype=torch.float32
        )

        self.target = torch.tensor(
            target,
            dtype=torch.float32
        )

        self.labels = torch.tensor(
            labels,
            dtype=torch.float32
        )


    def __len__(self):

        return len(self.labels)


    def __getitem__(self, index):

        return (
            self.source[index],
            self.target[index],
            self.labels[index]
        )


# ------------------------------------------------------------
# Create datasets
# ------------------------------------------------------------

train_dataset = CrossModalGANDataset(
    train_source,
    train_target,
    train_labels
)

validation_dataset = CrossModalGANDataset(
    validation_source,
    validation_target,
    validation_labels
)


print("=" * 100)
print("03.06 GAN TRAINING — DATASET OBJECTS")
print("=" * 100)

print(
    "Training dataset samples   :",
    len(train_dataset)
)

print(
    "Validation dataset samples :",
    len(validation_dataset)
)


# ------------------------------------------------------------
# Inspect one sample
# ------------------------------------------------------------

sample_source, sample_target, sample_label = (
    train_dataset[0]
)

print("\nFirst training sample:")
print(
    "Source shape :",
    tuple(sample_source.shape)
)

print(
    "Target shape :",
    tuple(sample_target.shape)
)

print(
    "Label        :",
    sample_label.item()
)


# ------------------------------------------------------------
# Safety checks
# ------------------------------------------------------------

assert sample_source.shape == (32,)
assert sample_target.shape == (32,)
assert sample_label.item() in (0.0, 1.0)


print("\n✓ GAN DATASET OBJECTS CREATED")

03.06 GAN TRAINING — DATASET OBJECTS
Training dataset samples   : 1776
Validation dataset samples : 448

First training sample:
Source shape : (32,)
Target shape : (32,)
Label        : 1.0

✓ GAN DATASET OBJECTS CREATED


In [12]:
# ============================================================
# CELL 12 — GAN DATALOADERS
# ============================================================

BATCH_SIZE = 32


# ------------------------------------------------------------
# Dedicated generator for reproducible training shuffling
# ------------------------------------------------------------

train_loader_generator = torch.Generator()

train_loader_generator.manual_seed(SEED)


# ------------------------------------------------------------
# Training DataLoader
# ------------------------------------------------------------

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=train_loader_generator
)


# ------------------------------------------------------------
# Validation DataLoader
# ------------------------------------------------------------

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)


print("=" * 100)
print("03.06 GAN TRAINING — DATALOADERS")
print("=" * 100)

print(
    "Batch size            :",
    BATCH_SIZE
)

print(
    "Training samples      :",
    len(train_dataset)
)

print(
    "Validation samples    :",
    len(validation_dataset)
)

print(
    "Training batches      :",
    len(train_loader)
)

print(
    "Validation batches    :",
    len(validation_loader)
)

print(
    "Training shuffle      : True"
)

print(
    "Validation shuffle    : False"
)

print(
    "DataLoader seed       :",
    SEED
)


assert len(train_loader) == 56
assert len(validation_loader) == 14


print("\n✓ GAN DATALOADERS CREATED")

03.06 GAN TRAINING — DATALOADERS
Batch size            : 32
Training samples      : 1776
Validation samples    : 448
Training batches      : 56
Validation batches    : 14
Training shuffle      : True
Validation shuffle    : False
DataLoader seed       : 42

✓ GAN DATALOADERS CREATED


In [13]:
# ============================================================
# CELL 13 — GAN LOSS FUNCTIONS
# ============================================================

adversarial_loss_fn = nn.BCEWithLogitsLoss()

semantic_loss_fn = nn.BCEWithLogitsLoss()


print("=" * 100)
print("03.06 GAN TRAINING — LOSS FUNCTIONS")
print("=" * 100)

print("\nAdversarial loss:")
print("  BCEWithLogitsLoss")

print("\nSemantic relationship loss:")
print("  BCEWithLogitsLoss")


print("\n" + "-" * 100)
print("LOSS DEFINITIONS")
print("-" * 100)

print("""
Discriminator real loss:
    BCEWithLogitsLoss(D(real_target), 1)

Discriminator fake loss:
    BCEWithLogitsLoss(D(generated_target), 0)

Generator adversarial loss:
    BCEWithLogitsLoss(D(generated_target), 1)

Semantic relationship loss:
    BCEWithLogitsLoss(
        Baseline(source, generated_target),
        pair_label
    )
""")


print("✓ GAN LOSS FUNCTIONS DEFINED")

03.06 GAN TRAINING — LOSS FUNCTIONS

Adversarial loss:
  BCEWithLogitsLoss

Semantic relationship loss:
  BCEWithLogitsLoss

----------------------------------------------------------------------------------------------------
LOSS DEFINITIONS
----------------------------------------------------------------------------------------------------

Discriminator real loss:
    BCEWithLogitsLoss(D(real_target), 1)

Discriminator fake loss:
    BCEWithLogitsLoss(D(generated_target), 0)

Generator adversarial loss:
    BCEWithLogitsLoss(D(generated_target), 1)

Semantic relationship loss:
    BCEWithLogitsLoss(
        Baseline(source, generated_target),
        pair_label
    )

✓ GAN LOSS FUNCTIONS DEFINED


In [14]:
# ============================================================
# CELL 14 — GENERATOR OBJECTIVE
# ============================================================

SEMANTIC_LOSS_WEIGHT = 1.0


print("=" * 100)
print("03.06 GAN TRAINING — GENERATOR OBJECTIVE")
print("=" * 100)

print(
    "Semantic loss weight (lambda) :",
    SEMANTIC_LOSS_WEIGHT
)

print("\nGenerator objective:")

print(
    "\nL_G = L_G_adv + lambda * L_semantic"
)

print(
    "\nWith lambda =",
    SEMANTIC_LOSS_WEIGHT
)

print(
    "L_G = L_G_adv +",
    SEMANTIC_LOSS_WEIGHT,
    "* L_semantic"
)


assert SEMANTIC_LOSS_WEIGHT > 0


print("\n✓ GENERATOR OBJECTIVE CONFIGURED")

03.06 GAN TRAINING — GENERATOR OBJECTIVE
Semantic loss weight (lambda) : 1.0

Generator objective:

L_G = L_G_adv + lambda * L_semantic

With lambda = 1.0
L_G = L_G_adv + 1.0 * L_semantic

✓ GENERATOR OBJECTIVE CONFIGURED


In [15]:
# ============================================================
# CELL 15 — GAN OPTIMIZERS
# ============================================================

GAN_LEARNING_RATE = 0.001


generator_optimizer = torch.optim.Adam(
    generator.parameters(),
    lr=GAN_LEARNING_RATE
)


discriminator_optimizer = torch.optim.Adam(
    discriminator.parameters(),
    lr=GAN_LEARNING_RATE
)


print("=" * 100)
print("03.06 GAN TRAINING — OPTIMIZER CONFIGURATION")
print("=" * 100)

print(
    "Generator optimizer      : Adam"
)

print(
    "Discriminator optimizer  : Adam"
)

print(
    "Learning rate            :",
    GAN_LEARNING_RATE
)


print("\nGenerator parameters optimized :",
      sum(
          p.numel()
          for p in generator.parameters()
      )
)

print(
    "Discriminator parameters optimized :",
    sum(
        p.numel()
        for p in discriminator.parameters()
    )
)


assert GAN_LEARNING_RATE > 0


print("\n✓ GAN OPTIMIZERS CREATED")

03.06 GAN TRAINING — OPTIMIZER CONFIGURATION
Generator optimizer      : Adam
Discriminator optimizer  : Adam
Learning rate            : 0.001

Generator parameters optimized : 20768
Discriminator parameters optimized : 4225

✓ GAN OPTIMIZERS CREATED


In [16]:
# ============================================================
# CELL 16 — SINGLE GAN TRAINING STEP SANITY CHECK
# ============================================================

print("=" * 100)
print("03.06 GAN TRAINING — SINGLE-STEP SANITY CHECK")
print("=" * 100)


# ------------------------------------------------------------
# Get one training batch
# ------------------------------------------------------------

source_batch, real_target_batch, label_batch = next(
    iter(train_loader)
)


source_batch = source_batch.to(device)
real_target_batch = real_target_batch.to(device)
label_batch = label_batch.to(device)


print("\nBATCH SHAPES")
print("-" * 70)

print(
    "Source batch shape      :",
    tuple(source_batch.shape)
)

print(
    "Real target batch shape :",
    tuple(real_target_batch.shape)
)

print(
    "Label batch shape       :",
    tuple(label_batch.shape)
)


# ------------------------------------------------------------
# Save copies of parameters BEFORE update
# ------------------------------------------------------------

generator_before = {
    name: parameter.detach().clone()
    for name, parameter in generator.named_parameters()
}

discriminator_before = {
    name: parameter.detach().clone()
    for name, parameter in discriminator.named_parameters()
}

baseline_before = {
    name: parameter.detach().clone()
    for name, parameter in baseline_model.named_parameters()
}


# ============================================================
# PART A — DISCRIMINATOR UPDATE
# ============================================================

discriminator_optimizer.zero_grad(
    set_to_none=True
)


# Generate target representation
generated_target = generator(
    source_batch
)


# Real target prediction
real_logits = discriminator(
    real_target_batch
)


# Fake target prediction
fake_logits = discriminator(
    generated_target.detach()
)


# Real target labels = 1
real_labels = torch.ones_like(
    real_logits
)


# Fake target labels = 0
fake_labels = torch.zeros_like(
    fake_logits
)


# Discriminator losses
discriminator_real_loss = adversarial_loss_fn(
    real_logits,
    real_labels
)

discriminator_fake_loss = adversarial_loss_fn(
    fake_logits,
    fake_labels
)


discriminator_loss = (
    discriminator_real_loss
    +
    discriminator_fake_loss
)


# Backpropagate discriminator
discriminator_loss.backward()


# Check discriminator gradients
discriminator_gradients_finite = all(
    parameter.grad is not None
    and torch.isfinite(parameter.grad).all()
    for parameter in discriminator.parameters()
)


if not discriminator_gradients_finite:

    raise RuntimeError(
        "Non-finite discriminator gradients detected."
    )


discriminator_optimizer.step()


# ============================================================
# PART B — GENERATOR UPDATE
# ============================================================

generator_optimizer.zero_grad(
    set_to_none=True
)


# Generate fresh target representation
generated_target = generator(
    source_batch
)


# ------------------------------------------------------------
# Generator adversarial objective
# ------------------------------------------------------------

generator_fake_logits = discriminator(
    generated_target
)


generator_real_labels = torch.ones_like(
    generator_fake_logits
)


generator_adversarial_loss = (
    adversarial_loss_fn(
        generator_fake_logits,
        generator_real_labels
    )
)


# ------------------------------------------------------------
# Semantic relationship objective
# ------------------------------------------------------------

semantic_logits = baseline_model(
    source_batch,
    generated_target
)


semantic_loss = semantic_loss_fn(
    semantic_logits,
    label_batch
)


# ------------------------------------------------------------
# Combined Generator loss
# ------------------------------------------------------------

generator_loss = (
    generator_adversarial_loss
    +
    SEMANTIC_LOSS_WEIGHT * semantic_loss
)


# ------------------------------------------------------------
# Backpropagate Generator
# ------------------------------------------------------------

generator_loss.backward()


# Check Generator gradients
generator_gradients_finite = all(
    parameter.grad is not None
    and torch.isfinite(parameter.grad).all()
    for parameter in generator.parameters()
)


if not generator_gradients_finite:

    raise RuntimeError(
        "Non-finite generator gradients detected."
    )


generator_optimizer.step()


# ============================================================
# PARAMETER CHANGE AUDIT
# ============================================================

generator_changed = any(
    not torch.equal(
        generator_before[name],
        parameter.detach()
    )
    for name, parameter in generator.named_parameters()
)


discriminator_changed = any(
    not torch.equal(
        discriminator_before[name],
        parameter.detach()
    )
    for name, parameter in discriminator.named_parameters()
)


baseline_unchanged = all(
    torch.equal(
        baseline_before[name],
        parameter.detach()
    )
    for name, parameter in baseline_model.named_parameters()
)


# ------------------------------------------------------------
# Finite loss checks
# ------------------------------------------------------------

all_losses_finite = all([
    torch.isfinite(discriminator_real_loss).item(),
    torch.isfinite(discriminator_fake_loss).item(),
    torch.isfinite(discriminator_loss).item(),
    torch.isfinite(generator_adversarial_loss).item(),
    torch.isfinite(semantic_loss).item(),
    torch.isfinite(generator_loss).item()
])


# ============================================================
# PRINT RESULTS
# ============================================================

print("\n" + "-" * 100)
print("LOSS RESULTS")
print("-" * 100)

print(
    "Discriminator real loss       :",
    discriminator_real_loss.item()
)

print(
    "Discriminator fake loss       :",
    discriminator_fake_loss.item()
)

print(
    "Total discriminator loss      :",
    discriminator_loss.item()
)

print(
    "Generator adversarial loss    :",
    generator_adversarial_loss.item()
)

print(
    "Semantic relationship loss    :",
    semantic_loss.item()
)

print(
    "Total generator loss          :",
    generator_loss.item()
)


print("\n" + "-" * 100)
print("GRADIENT / PARAMETER AUDIT")
print("-" * 100)

print(
    "Generator gradients finite    :",
    generator_gradients_finite
)

print(
    "Discriminator gradients finite:",
    discriminator_gradients_finite
)

print(
    "All losses finite             :",
    all_losses_finite
)

print(
    "Generator weights changed     :",
    generator_changed
)

print(
    "Discriminator weights changed :",
    discriminator_changed
)

print(
    "Baseline weights unchanged    :",
    baseline_unchanged
)


# ------------------------------------------------------------
# Final safety assertions
# ------------------------------------------------------------

assert all_losses_finite
assert generator_gradients_finite
assert discriminator_gradients_finite

assert generator_changed
assert discriminator_changed

assert baseline_unchanged


print("\n" + "=" * 100)
print("✓ SINGLE GAN TRAINING STEP PASSED")
print("=" * 100)

03.06 GAN TRAINING — SINGLE-STEP SANITY CHECK

BATCH SHAPES
----------------------------------------------------------------------
Source batch shape      : (32, 32)
Real target batch shape : (32, 32)
Label batch shape       : (32,)

----------------------------------------------------------------------------------------------------
LOSS RESULTS
----------------------------------------------------------------------------------------------------
Discriminator real loss       : 0.696628212928772
Discriminator fake loss       : 0.7772818803787231
Total discriminator loss      : 1.4739100933074951
Generator adversarial loss    : 0.6162927746772766
Semantic relationship loss    : 0.7662304043769836
Total generator loss          : 1.3825231790542603

----------------------------------------------------------------------------------------------------
GRADIENT / PARAMETER AUDIT
----------------------------------------------------------------------------------------------------
Generator gradie

In [17]:
# ============================================================
# CELL 17 — CLEAN GAN TRAINING RESET
# ============================================================

print("=" * 100)
print("03.06 GAN TRAINING — CLEAN TRAINING RESET")
print("=" * 100)


# ------------------------------------------------------------
# Reset random seed
# ------------------------------------------------------------

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():

    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)


# ------------------------------------------------------------
# Recreate Generator
# ------------------------------------------------------------

generator = Generator(
    input_dim=32,
    output_dim=32
).to(device)


# ------------------------------------------------------------
# Recreate Discriminator
# ------------------------------------------------------------

discriminator = Discriminator(
    input_dim=32
).to(device)


# ------------------------------------------------------------
# Recreate optimizers
# ------------------------------------------------------------

generator_optimizer = torch.optim.Adam(
    generator.parameters(),
    lr=GAN_LEARNING_RATE
)

discriminator_optimizer = torch.optim.Adam(
    discriminator.parameters(),
    lr=GAN_LEARNING_RATE
)


# ------------------------------------------------------------
# Make sure semantic baseline remains frozen
# ------------------------------------------------------------

baseline_model.eval()

for parameter in baseline_model.parameters():

    parameter.requires_grad = False


# ------------------------------------------------------------
# Recreate DataLoader generator
# ------------------------------------------------------------

train_loader_generator = torch.Generator()

train_loader_generator.manual_seed(SEED)


# ------------------------------------------------------------
# Recreate training DataLoader
# ------------------------------------------------------------

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=train_loader_generator
)


# ------------------------------------------------------------
# Verify trainable/frozen parameters
# ------------------------------------------------------------

generator_trainable = sum(
    p.numel()
    for p in generator.parameters()
    if p.requires_grad
)

discriminator_trainable = sum(
    p.numel()
    for p in discriminator.parameters()
    if p.requires_grad
)

baseline_trainable = sum(
    p.numel()
    for p in baseline_model.parameters()
    if p.requires_grad
)


print("\nGenerator trainable parameters     :",
      generator_trainable)

print("Discriminator trainable parameters :",
      discriminator_trainable)

print("Baseline trainable parameters      :",
      baseline_trainable)

print("DataLoader seed                    :",
      SEED)


# ------------------------------------------------------------
# Safety assertions
# ------------------------------------------------------------

assert generator_trainable == 20768
assert discriminator_trainable == 4225
assert baseline_trainable == 0


print("\n" + "=" * 100)
print("✓ CLEAN GAN TRAINING RESET PASSED")
print("✓ READY FOR OFFICIAL GAN TRAINING")
print("=" * 100)

03.06 GAN TRAINING — CLEAN TRAINING RESET

Generator trainable parameters     : 20768
Discriminator trainable parameters : 4225
Baseline trainable parameters      : 0
DataLoader seed                    : 42

✓ CLEAN GAN TRAINING RESET PASSED
✓ READY FOR OFFICIAL GAN TRAINING


In [18]:
# ============================================================
# CELL 18 — OFFICIAL GAN TRAINING CONFIGURATION
# ============================================================

NUM_EPOCHS = 50

GAN_LEARNING_RATE = 0.001

SEMANTIC_LOSS_WEIGHT = 1.0

BATCH_SIZE = 32


# ------------------------------------------------------------
# Checkpoint / output directories
# ------------------------------------------------------------

gan_training_root = (
    official_phase3_root
    / "03_06_gan_training"
)

gan_checkpoint_dir = (
    gan_training_root
    / "checkpoints"
)

gan_report_dir = (
    gan_training_root
    / "reports"
)

gan_figure_dir = (
    gan_training_root
    / "figures"
)


# ------------------------------------------------------------
# Create directories
# ------------------------------------------------------------

gan_checkpoint_dir.mkdir(
    parents=True,
    exist_ok=True
)

gan_report_dir.mkdir(
    parents=True,
    exist_ok=True
)

gan_figure_dir.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# Checkpoint paths
# ------------------------------------------------------------

best_generator_path = (
    gan_checkpoint_dir
    / "03_06_gan_best_generator.pt"
)

best_discriminator_path = (
    gan_checkpoint_dir
    / "03_06_gan_best_discriminator.pt"
)

final_generator_path = (
    gan_checkpoint_dir
    / "03_06_gan_final_generator.pt"
)

final_discriminator_path = (
    gan_checkpoint_dir
    / "03_06_gan_final_discriminator.pt"
)

training_history_path = (
    gan_checkpoint_dir
    / "03_06_gan_training_history.pt"
)

training_config_path = (
    gan_report_dir
    / "03_06_gan_training_configuration.json"
)


print("=" * 100)
print("03.06 GAN TRAINING — OFFICIAL TRAINING CONFIGURATION")
print("=" * 100)

print("\nNumber of epochs          :", NUM_EPOCHS)
print("Batch size                :", BATCH_SIZE)
print("Learning rate             :", GAN_LEARNING_RATE)
print("Semantic loss weight      :", SEMANTIC_LOSS_WEIGHT)
print("Random seed               :", SEED)

print("\nPrimary model-selection metric:")
print("Validation semantic ROC-AUC")

print("\nBaseline reference ROC-AUC:")
print("0.662667")

print("\nCheckpoint directory:")
print(gan_checkpoint_dir)

print("\nBest Generator:")
print(best_generator_path)

print("\nBest Discriminator:")
print(best_discriminator_path)

print("\nFinal Generator:")
print(final_generator_path)

print("\nFinal Discriminator:")
print(final_discriminator_path)

print("\nTraining history:")
print(training_history_path)

print("\nTraining configuration:")
print(training_config_path)


print("\n" + "=" * 100)
print("✓ OFFICIAL GAN TRAINING CONFIGURATION READY")
print("=" * 100)

03.06 GAN TRAINING — OFFICIAL TRAINING CONFIGURATION

Number of epochs          : 50
Batch size                : 32
Learning rate             : 0.001
Semantic loss weight      : 1.0
Random seed               : 42

Primary model-selection metric:
Validation semantic ROC-AUC

Baseline reference ROC-AUC:
0.662667

Checkpoint directory:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\03_cross_modal_learning\03_06_gan_training\checkpoints

Best Generator:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\03_cross_modal_learning\03_06_gan_training\checkpoints\03_06_gan_best_generator.pt

Best Discriminator:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\03_cross_modal_learning\03_06_gan_training\checkpoints\03_06_gan_best_discriminator.pt

Final Generator:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\03_cross

In [19]:
# ============================================================
# CELL 19 — SAVE OFFICIAL GAN TRAINING CONFIGURATION
# ============================================================

training_config = {
    "phase": "03.06",
    "experiment": "GAN Training",

    "input_dimension": 32,
    "target_dimension": 32,

    "generator_architecture": [
        32,
        64,
        128,
        64,
        32
    ],

    "discriminator_architecture": [
        32,
        64,
        32,
        1
    ],

    "semantic_relationship_architecture": [
        128,
        128,
        64,
        1
    ],

    "epochs": NUM_EPOCHS,

    "batch_size": BATCH_SIZE,

    "learning_rate": GAN_LEARNING_RATE,

    "optimizer": "Adam",

    "loss_function": "BCEWithLogitsLoss",

    "semantic_loss_weight":
        SEMANTIC_LOSS_WEIGHT,

    "random_seed": SEED,

    "train_samples": len(train_dataset),

    "validation_samples": len(validation_dataset),

    "train_positive": int(
        np.sum(train_labels == 1)
    ),

    "train_negative": int(
        np.sum(train_labels == 0)
    ),

    "validation_positive": int(
        np.sum(validation_labels == 1)
    ),

    "validation_negative": int(
        np.sum(validation_labels == 0)
    ),

    "baseline_reference_roc_auc":
        0.662667,

    "primary_selection_metric":
        "validation_semantic_roc_auc",

    "dataset":
        "Phase 02.08 representation-aware split",

    "training_type":
        "unpaired class-consistent cross-modal representation learning"
}


with open(
    training_config_path,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        training_config,
        file,
        indent=4
    )


print("=" * 100)
print("03.06 GAN TRAINING — CONFIGURATION SAVED")
print("=" * 100)

print("\nConfiguration file:")
print(training_config_path)

print(
    "\nFile exists:",
    training_config_path.exists()
)

print("\n✓ TRAINING CONFIGURATION SAVED")

03.06 GAN TRAINING — CONFIGURATION SAVED

Configuration file:
C:\Users\shasw\Documents\GitHub\GAN-Enhanced-Multimodal-ASD-Screening\06_cross_modal_learning\03_cross_modal_learning\03_06_gan_training\reports\03_06_gan_training_configuration.json

File exists: True

✓ TRAINING CONFIGURATION SAVED


In [20]:
# ============================================================
# CELL 20 — FINAL PRE-TRAINING SAFETY AUDIT
# ============================================================

print("=" * 100)
print("03.06 GAN TRAINING — FINAL PRE-TRAINING SAFETY AUDIT")
print("=" * 100)


# ------------------------------------------------------------
# Verify model modes
# ------------------------------------------------------------

generator.train()
discriminator.train()

baseline_model.eval()


# ------------------------------------------------------------
# Verify baseline remains frozen
# ------------------------------------------------------------

baseline_trainable_parameters = sum(
    p.numel()
    for p in baseline_model.parameters()
    if p.requires_grad
)


# ------------------------------------------------------------
# Verify GAN networks are trainable
# ------------------------------------------------------------

generator_trainable_parameters = sum(
    p.numel()
    for p in generator.parameters()
    if p.requires_grad
)

discriminator_trainable_parameters = sum(
    p.numel()
    for p in discriminator.parameters()
    if p.requires_grad
)


# ------------------------------------------------------------
# Verify dataset sizes
# ------------------------------------------------------------

assert len(train_dataset) == 1776
assert len(validation_dataset) == 448


# ------------------------------------------------------------
# Verify batch configuration
# ------------------------------------------------------------

assert BATCH_SIZE == 32
assert len(train_loader) == 56
assert len(validation_loader) == 14


# ------------------------------------------------------------
# Verify parameter counts
# ------------------------------------------------------------

assert generator_trainable_parameters == 20768

assert discriminator_trainable_parameters == 4225

assert baseline_trainable_parameters == 0


# ------------------------------------------------------------
# Verify required files
# ------------------------------------------------------------

assert baseline_checkpoint_path.exists()

assert training_config_path.exists()


print("\nDATASET")
print("-" * 70)

print(
    "Training samples       :",
    len(train_dataset)
)

print(
    "Validation samples     :",
    len(validation_dataset)
)


print("\nMODEL PARAMETERS")
print("-" * 70)

print(
    "Generator trainable    :",
    generator_trainable_parameters
)

print(
    "Discriminator trainable:",
    discriminator_trainable_parameters
)

print(
    "Baseline trainable     :",
    baseline_trainable_parameters
)


print("\nMODEL MODES")
print("-" * 70)

print(
    "Generator training mode      :",
    generator.training
)

print(
    "Discriminator training mode  :",
    discriminator.training
)

print(
    "Baseline evaluation mode     :",
    not baseline_model.training
)


print("\nDEPENDENCY CHECK")
print("-" * 70)

print(
    "Baseline checkpoint exists    :",
    baseline_checkpoint_path.exists()
)

print(
    "Training config exists        :",
    training_config_path.exists()
)


print("\n" + "=" * 100)
print("✓ FINAL PRE-TRAINING SAFETY AUDIT PASSED")
print("✓ READY TO START OFFICIAL GAN TRAINING")
print("=" * 100)

03.06 GAN TRAINING — FINAL PRE-TRAINING SAFETY AUDIT

DATASET
----------------------------------------------------------------------
Training samples       : 1776
Validation samples     : 448

MODEL PARAMETERS
----------------------------------------------------------------------
Generator trainable    : 20768
Discriminator trainable: 4225
Baseline trainable     : 0

MODEL MODES
----------------------------------------------------------------------
Generator training mode      : True
Discriminator training mode  : True
Baseline evaluation mode     : True

DEPENDENCY CHECK
----------------------------------------------------------------------
Baseline checkpoint exists    : True
Training config exists        : True

✓ FINAL PRE-TRAINING SAFETY AUDIT PASSED
✓ READY TO START OFFICIAL GAN TRAINING


In [21]:
# ==================================================================================================
# CELL 21 — VALIDATION EVALUATION FUNCTION
# ==================================================================================================

import numpy as np
import torch
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

print("=" * 100)
print("03.06 GAN TRAINING — VALIDATION EVALUATION FUNCTION")
print("=" * 100)


def evaluate_gan(
    generator,
    baseline_model,
    validation_loader,
    semantic_loss_fn,
    device
):
    """
    Evaluate the Generator on the fixed validation set.

    The Generator creates a target-like representation from the source
    representation. The frozen baseline relationship model then evaluates
    whether the generated representation preserves the expected semantic
    relationship between the source and target.

    Returns:
        Dictionary containing validation loss and classification metrics.
    """

    generator.eval()
    baseline_model.eval()

    all_labels = []
    all_probabilities = []
    all_predictions = []

    total_loss = 0.0
    total_samples = 0

    with torch.no_grad():

        for source, real_target, labels in validation_loader:

            source = source.to(device)
            labels = labels.to(device)

            # ------------------------------------------------------------------
            # Generate target-like representation
            # ------------------------------------------------------------------
            generated_target = generator(source)

            # ------------------------------------------------------------------
            # Evaluate generated cross-modal relationship
            # ------------------------------------------------------------------
            semantic_logits = baseline_model(
                source,
                generated_target
            )

            semantic_loss = semantic_loss_fn(
                semantic_logits,
                labels
            )

            # ------------------------------------------------------------------
            # Convert logits to probabilities
            # ------------------------------------------------------------------
            probabilities = torch.sigmoid(semantic_logits)

            predictions = (probabilities >= 0.5).long()

            # ------------------------------------------------------------------
            # Store results
            # ------------------------------------------------------------------
            batch_size = source.size(0)

            total_loss += semantic_loss.item() * batch_size
            total_samples += batch_size

            all_labels.extend(
                labels.cpu().numpy().tolist()
            )

            all_probabilities.extend(
                probabilities.cpu().numpy().tolist()
            )

            all_predictions.extend(
                predictions.cpu().numpy().tolist()
            )

    # ----------------------------------------------------------------------------------------------
    # Convert collected results to NumPy arrays
    # ----------------------------------------------------------------------------------------------

    all_labels = np.asarray(all_labels)
    all_probabilities = np.asarray(all_probabilities)
    all_predictions = np.asarray(all_predictions)

    # ----------------------------------------------------------------------------------------------
    # Calculate metrics
    # ----------------------------------------------------------------------------------------------

    validation_loss = total_loss / total_samples

    validation_accuracy = accuracy_score(
        all_labels,
        all_predictions
    )

    validation_balanced_accuracy = balanced_accuracy_score(
        all_labels,
        all_predictions
    )

    validation_precision = precision_score(
        all_labels,
        all_predictions,
        zero_division=0
    )

    validation_recall = recall_score(
        all_labels,
        all_predictions,
        zero_division=0
    )

    validation_f1 = f1_score(
        all_labels,
        all_predictions,
        zero_division=0
    )

    validation_roc_auc = roc_auc_score(
        all_labels,
        all_probabilities
    )

    # ----------------------------------------------------------------------------------------------
    # Return all validation metrics
    # ----------------------------------------------------------------------------------------------

    metrics = {
        "loss": float(validation_loss),
        "accuracy": float(validation_accuracy),
        "balanced_accuracy": float(validation_balanced_accuracy),
        "precision": float(validation_precision),
        "recall": float(validation_recall),
        "f1": float(validation_f1),
        "roc_auc": float(validation_roc_auc)
    }

    return metrics


print("✓ Validation evaluation function defined successfully")
print()
print("Metrics:")
print("  • Validation Loss")
print("  • Accuracy")
print("  • Balanced Accuracy")
print("  • Precision")
print("  • Recall")
print("  • F1 Score")
print("  • ROC-AUC")
print()
print("✓ Validation uses no gradient computation")
print("✓ Baseline relationship model remains frozen")
print("✓ Function definition passed")

03.06 GAN TRAINING — VALIDATION EVALUATION FUNCTION
✓ Validation evaluation function defined successfully

Metrics:
  • Validation Loss
  • Accuracy
  • Balanced Accuracy
  • Precision
  • Recall
  • F1 Score
  • ROC-AUC

✓ Validation uses no gradient computation
✓ Baseline relationship model remains frozen
✓ Function definition passed


In [23]:
# ==================================================================================================
# CELL 22 — OFFICIAL GAN TRAINING LOOP
# ==================================================================================================

import copy
import time
import numpy as np
import torch

print("=" * 100)
print("03.06 GAN TRAINING — OFFICIAL TRAINING")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# Official baseline reference
# --------------------------------------------------------------------------------------------------
# This is the verified ROC-AUC from the reproducible official 03.04 baseline training.
# The baseline checkpoint itself is NOT modified.

BASELINE_REFERENCE_ROC_AUC = 0.662667

# --------------------------------------------------------------------------------------------------
# Reproducibility reset
# --------------------------------------------------------------------------------------------------

torch.manual_seed(SEED)
np.random.seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# --------------------------------------------------------------------------------------------------
# Recreate DataLoaders deterministically
# --------------------------------------------------------------------------------------------------

train_loader_generator = torch.Generator()
train_loader_generator.manual_seed(SEED)

train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=train_loader_generator
)

validation_loader = torch.utils.data.DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

# --------------------------------------------------------------------------------------------------
# Reset Generator and Discriminator
# --------------------------------------------------------------------------------------------------

generator = Generator().to(device)
discriminator = Discriminator().to(device)

# --------------------------------------------------------------------------------------------------
# Recreate optimizers
# --------------------------------------------------------------------------------------------------

generator_optimizer = torch.optim.Adam(
    generator.parameters(),
    lr=GAN_LEARNING_RATE
)

discriminator_optimizer = torch.optim.Adam(
    discriminator.parameters(),
    lr=GAN_LEARNING_RATE
)

# --------------------------------------------------------------------------------------------------
# Ensure baseline remains frozen
# --------------------------------------------------------------------------------------------------

baseline_model.eval()

for parameter in baseline_model.parameters():
    parameter.requires_grad = False

# --------------------------------------------------------------------------------------------------
# Training history
# --------------------------------------------------------------------------------------------------

training_history = {
    "epoch": [],
    "generator_total_loss": [],
    "generator_adversarial_loss": [],
    "generator_semantic_loss": [],
    "discriminator_total_loss": [],
    "discriminator_real_loss": [],
    "discriminator_fake_loss": [],
    "validation_loss": [],
    "validation_accuracy": [],
    "validation_balanced_accuracy": [],
    "validation_precision": [],
    "validation_recall": [],
    "validation_f1": [],
    "validation_roc_auc": []
}

# --------------------------------------------------------------------------------------------------
# Best-model tracking
# --------------------------------------------------------------------------------------------------

best_validation_roc_auc = -np.inf
best_epoch = None

best_generator_state = None
best_discriminator_state = None

# --------------------------------------------------------------------------------------------------
# Training start
# --------------------------------------------------------------------------------------------------

training_start_time = time.time()

print()
print("TRAINING CONFIGURATION")
print("-" * 70)
print(f"Epochs                  : {NUM_EPOCHS}")
print(f"Batch size              : {BATCH_SIZE}")
print(f"Learning rate           : {GAN_LEARNING_RATE}")
print(f"Semantic loss weight    : {SEMANTIC_LOSS_WEIGHT}")
print(f"Training batches/epoch  : {len(train_loader)}")
print(f"Validation batches      : {len(validation_loader)}")
print(f"Device                  : {device}")
print(f"Baseline reference AUC  : {BASELINE_REFERENCE_ROC_AUC:.6f}")
print()
print("Starting official GAN training...")
print("-" * 100)

# ==================================================================================================
# EPOCH LOOP
# ==================================================================================================

for epoch in range(1, NUM_EPOCHS + 1):

    generator.train()
    discriminator.train()
    baseline_model.eval()

    # ----------------------------------------------------------------------------------------------
    # Epoch accumulators
    # ----------------------------------------------------------------------------------------------

    epoch_generator_total = 0.0
    epoch_generator_adversarial = 0.0
    epoch_generator_semantic = 0.0

    epoch_discriminator_total = 0.0
    epoch_discriminator_real = 0.0
    epoch_discriminator_fake = 0.0

    epoch_samples = 0

    # ==============================================================================================
    # BATCH LOOP
    # ==============================================================================================

    for source, real_target, labels in train_loader:

        source = source.to(device)
        real_target = real_target.to(device)
        labels = labels.to(device)

        current_batch_size = source.size(0)

        # ==========================================================================================
        # 1. TRAIN DISCRIMINATOR
        # ==========================================================================================

        discriminator_optimizer.zero_grad()

        generated_target = generator(source)

        real_logits = discriminator(real_target)

        # Detach prevents discriminator training from updating Generator
        fake_logits = discriminator(
            generated_target.detach()
        )

        real_labels = torch.ones_like(real_logits)
        fake_labels = torch.zeros_like(fake_logits)

        discriminator_real_loss = adversarial_loss_fn(
            real_logits,
            real_labels
        )

        discriminator_fake_loss = adversarial_loss_fn(
            fake_logits,
            fake_labels
        )

        discriminator_total_loss = (
            discriminator_real_loss +
            discriminator_fake_loss
        )

        discriminator_total_loss.backward()

        discriminator_optimizer.step()

        # ==========================================================================================
        # 2. TRAIN GENERATOR
        # ==========================================================================================

        generator_optimizer.zero_grad()

        generated_target = generator(source)

        # Adversarial objective
        fake_logits_for_generator = discriminator(
            generated_target
        )

        generator_real_labels = torch.ones_like(
            fake_logits_for_generator
        )

        generator_adversarial_loss = adversarial_loss_fn(
            fake_logits_for_generator,
            generator_real_labels
        )

        # Semantic relationship objective
        semantic_logits = baseline_model(
            source,
            generated_target
        )

        generator_semantic_loss = semantic_loss_fn(
            semantic_logits,
            labels
        )

        # Total Generator objective
        generator_total_loss = (
            generator_adversarial_loss +
            SEMANTIC_LOSS_WEIGHT * generator_semantic_loss
        )

        generator_total_loss.backward()

        generator_optimizer.step()

        # ==========================================================================================
        # 3. ACCUMULATE LOSSES
        # ==========================================================================================

        epoch_generator_total += (
            generator_total_loss.item() * current_batch_size
        )

        epoch_generator_adversarial += (
            generator_adversarial_loss.item() * current_batch_size
        )

        epoch_generator_semantic += (
            generator_semantic_loss.item() * current_batch_size
        )

        epoch_discriminator_total += (
            discriminator_total_loss.item() * current_batch_size
        )

        epoch_discriminator_real += (
            discriminator_real_loss.item() * current_batch_size
        )

        epoch_discriminator_fake += (
            discriminator_fake_loss.item() * current_batch_size
        )

        epoch_samples += current_batch_size

    # ==============================================================================================
    # 4. AVERAGE TRAINING LOSSES
    # ==============================================================================================

    avg_generator_total = (
        epoch_generator_total / epoch_samples
    )

    avg_generator_adversarial = (
        epoch_generator_adversarial / epoch_samples
    )

    avg_generator_semantic = (
        epoch_generator_semantic / epoch_samples
    )

    avg_discriminator_total = (
        epoch_discriminator_total / epoch_samples
    )

    avg_discriminator_real = (
        epoch_discriminator_real / epoch_samples
    )

    avg_discriminator_fake = (
        epoch_discriminator_fake / epoch_samples
    )

    # ==============================================================================================
    # 5. VALIDATION
    # ==============================================================================================

    validation_metrics = evaluate_gan(
        generator=generator,
        baseline_model=baseline_model,
        validation_loader=validation_loader,
        semantic_loss_fn=semantic_loss_fn,
        device=device
    )

    # ==============================================================================================
    # 6. STORE HISTORY
    # ==============================================================================================

    training_history["epoch"].append(epoch)

    training_history["generator_total_loss"].append(
        avg_generator_total
    )

    training_history["generator_adversarial_loss"].append(
        avg_generator_adversarial
    )

    training_history["generator_semantic_loss"].append(
        avg_generator_semantic
    )

    training_history["discriminator_total_loss"].append(
        avg_discriminator_total
    )

    training_history["discriminator_real_loss"].append(
        avg_discriminator_real
    )

    training_history["discriminator_fake_loss"].append(
        avg_discriminator_fake
    )

    training_history["validation_loss"].append(
        validation_metrics["loss"]
    )

    training_history["validation_accuracy"].append(
        validation_metrics["accuracy"]
    )

    training_history["validation_balanced_accuracy"].append(
        validation_metrics["balanced_accuracy"]
    )

    training_history["validation_precision"].append(
        validation_metrics["precision"]
    )

    training_history["validation_recall"].append(
        validation_metrics["recall"]
    )

    training_history["validation_f1"].append(
        validation_metrics["f1"]
    )

    training_history["validation_roc_auc"].append(
        validation_metrics["roc_auc"]
    )

    # ==============================================================================================
    # 7. BEST MODEL CHECKPOINT
    # ==============================================================================================

    current_validation_roc_auc = validation_metrics["roc_auc"]

    if current_validation_roc_auc > best_validation_roc_auc:

        best_validation_roc_auc = current_validation_roc_auc
        best_epoch = epoch

        best_generator_state = copy.deepcopy(
            generator.state_dict()
        )

        best_discriminator_state = copy.deepcopy(
            discriminator.state_dict()
        )

        # Generator checkpoint
        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": best_generator_state,
                "optimizer_state_dict": generator_optimizer.state_dict(),
                "validation_roc_auc": best_validation_roc_auc,
                "baseline_reference_roc_auc": BASELINE_REFERENCE_ROC_AUC,
                "semantic_loss_weight": SEMANTIC_LOSS_WEIGHT,
                "seed": SEED
            },
            gan_checkpoint_dir / "03_06_gan_best_generator.pt"
        )

        # Discriminator checkpoint
        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": best_discriminator_state,
                "optimizer_state_dict": discriminator_optimizer.state_dict(),
                "validation_roc_auc": best_validation_roc_auc,
                "seed": SEED
            },
            gan_checkpoint_dir / "03_06_gan_best_discriminator.pt"
        )

        best_marker = "  ← BEST"

    else:
        best_marker = ""

    # ==============================================================================================
    # 8. PRINT EPOCH RESULT
    # ==============================================================================================

    print(
        f"Epoch {epoch:02d}/{NUM_EPOCHS} | "
        f"G: {avg_generator_total:.4f} | "
        f"D: {avg_discriminator_total:.4f} | "
        f"Val Loss: {validation_metrics['loss']:.4f} | "
        f"Val AUC: {current_validation_roc_auc:.4f}"
        f"{best_marker}"
    )

# ==================================================================================================
# TRAINING COMPLETE
# ==================================================================================================

training_time_seconds = time.time() - training_start_time

print()
print("=" * 100)
print("03.06 GAN TRAINING — TRAINING COMPLETE")
print("=" * 100)

print(f"Total training time      : {training_time_seconds:.2f} seconds")
print(f"Epochs completed         : {NUM_EPOCHS}")
print(f"Best validation ROC-AUC  : {best_validation_roc_auc:.6f}")
print(f"Best epoch               : {best_epoch}")
print(f"Baseline reference AUC   : {BASELINE_REFERENCE_ROC_AUC:.6f}")

if best_validation_roc_auc > BASELINE_REFERENCE_ROC_AUC:
    print("GAN vs baseline          : GAN BEST AUC IS HIGHER")
elif best_validation_roc_auc < BASELINE_REFERENCE_ROC_AUC:
    print("GAN vs baseline          : GAN BEST AUC IS LOWER")
else:
    print("GAN vs baseline          : GAN BEST AUC IS EQUAL")

print()
print("✓ Best Generator checkpoint saved")
print("✓ Best Discriminator checkpoint saved")
print("✓ Training history collected")
print("✓ Baseline model remained frozen")
print()
print("Official GAN training loop completed.")

03.06 GAN TRAINING — OFFICIAL TRAINING

TRAINING CONFIGURATION
----------------------------------------------------------------------
Epochs                  : 50
Batch size              : 32
Learning rate           : 0.001
Semantic loss weight    : 1.0
Training batches/epoch  : 56
Validation batches      : 14
Device                  : cpu
Baseline reference AUC  : 0.662667

Starting official GAN training...
----------------------------------------------------------------------------------------------------
Epoch 01/50 | G: 1.3912 | D: 0.8606 | Val Loss: 0.6970 | Val AUC: 0.5000  ← BEST
Epoch 02/50 | G: 1.6591 | D: 0.5903 | Val Loss: 0.7059 | Val AUC: 0.5000
Epoch 03/50 | G: 2.0606 | D: 0.4925 | Val Loss: 0.7099 | Val AUC: 0.5000  ← BEST
Epoch 04/50 | G: 2.7366 | D: 0.3649 | Val Loss: 0.6980 | Val AUC: 0.5000
Epoch 05/50 | G: 3.4521 | D: 0.3730 | Val Loss: 0.7472 | Val AUC: 0.5000  ← BEST
Epoch 06/50 | G: 4.1686 | D: 0.4218 | Val Loss: 0.8316 | Val AUC: 0.5000
Epoch 07/50 | G: 3.0532 |

In [24]:
# ==================================================================================================
# CELL 23 — GAN TRAINING DIAGNOSTIC
# ==================================================================================================

import numpy as np
import torch
from sklearn.metrics import roc_auc_score

print("=" * 100)
print("03.06 GAN TRAINING — POST-TRAINING DIAGNOSTIC")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# Load the best Generator checkpoint from the official run
# --------------------------------------------------------------------------------------------------

best_generator_checkpoint_path = (
    gan_checkpoint_dir / "03_06_gan_best_generator.pt"
)

best_generator_checkpoint = torch.load(
    best_generator_checkpoint_path,
    map_location=device
)

diagnostic_generator = Generator().to(device)

diagnostic_generator.load_state_dict(
    best_generator_checkpoint["model_state_dict"],
    strict=True
)

diagnostic_generator.eval()

# --------------------------------------------------------------------------------------------------
# Ensure baseline remains frozen
# --------------------------------------------------------------------------------------------------

baseline_model.eval()

for parameter in baseline_model.parameters():
    parameter.requires_grad = False

# --------------------------------------------------------------------------------------------------
# Collect validation outputs
# --------------------------------------------------------------------------------------------------

all_source = []
all_real_target = []
all_generated_target = []
all_labels = []
all_semantic_logits = []
all_semantic_probabilities = []

with torch.no_grad():

    for source, real_target, labels in validation_loader:

        source = source.to(device)
        real_target = real_target.to(device)
        labels = labels.to(device)

        generated_target = diagnostic_generator(source)

        semantic_logits = baseline_model(
            source,
            generated_target
        )

        semantic_probabilities = torch.sigmoid(
            semantic_logits
        )

        all_source.append(
            source.cpu()
        )

        all_real_target.append(
            real_target.cpu()
        )

        all_generated_target.append(
            generated_target.cpu()
        )

        all_labels.append(
            labels.cpu()
        )

        all_semantic_logits.append(
            semantic_logits.cpu()
        )

        all_semantic_probabilities.append(
            semantic_probabilities.cpu()
        )

# --------------------------------------------------------------------------------------------------
# Concatenate validation results
# --------------------------------------------------------------------------------------------------

all_source = torch.cat(all_source, dim=0)
all_real_target = torch.cat(all_real_target, dim=0)
all_generated_target = torch.cat(all_generated_target, dim=0)
all_labels = torch.cat(all_labels, dim=0)

all_semantic_logits = torch.cat(
    all_semantic_logits,
    dim=0
)

all_semantic_probabilities = torch.cat(
    all_semantic_probabilities,
    dim=0
)

# --------------------------------------------------------------------------------------------------
# Convert to NumPy
# --------------------------------------------------------------------------------------------------

labels_np = all_labels.numpy()

logits_np = all_semantic_logits.numpy()
probabilities_np = all_semantic_probabilities.numpy()

# --------------------------------------------------------------------------------------------------
# Basic statistics
# --------------------------------------------------------------------------------------------------

generated_mean = all_generated_target.mean().item()
generated_std = all_generated_target.std().item()
generated_min = all_generated_target.min().item()
generated_max = all_generated_target.max().item()

real_mean = all_real_target.mean().item()
real_std = all_real_target.std().item()
real_min = all_real_target.min().item()
real_max = all_real_target.max().item()

logit_mean = logits_np.mean()
logit_std = logits_np.std()
logit_min = logits_np.min()
logit_max = logits_np.max()

probability_mean = probabilities_np.mean()
probability_std = probabilities_np.std()
probability_min = probabilities_np.min()
probability_max = probabilities_np.max()

# --------------------------------------------------------------------------------------------------
# Label distribution
# --------------------------------------------------------------------------------------------------

unique_labels, label_counts = np.unique(
    labels_np,
    return_counts=True
)

# --------------------------------------------------------------------------------------------------
# Number of unique generated vectors
# --------------------------------------------------------------------------------------------------

generated_np = all_generated_target.numpy()

unique_generated = np.unique(
    np.round(generated_np, decimals=6),
    axis=0
).shape[0]

# --------------------------------------------------------------------------------------------------
# Calculate diagnostic AUC independently
# --------------------------------------------------------------------------------------------------

diagnostic_auc = roc_auc_score(
    labels_np,
    probabilities_np
)

# --------------------------------------------------------------------------------------------------
# Print results
# --------------------------------------------------------------------------------------------------

print()
print("VALIDATION LABEL DISTRIBUTION")
print("-" * 70)

for label_value, count in zip(unique_labels, label_counts):
    print(
        f"Label {int(label_value)} : {int(count)} samples"
    )

print()
print("REAL TARGET REPRESENTATIONS")
print("-" * 70)
print(f"Mean                  : {real_mean:.8f}")
print(f"Std                   : {real_std:.8f}")
print(f"Minimum               : {real_min:.8f}")
print(f"Maximum               : {real_max:.8f}")

print()
print("GENERATED TARGET REPRESENTATIONS")
print("-" * 70)
print(f"Mean                  : {generated_mean:.8f}")
print(f"Std                   : {generated_std:.8f}")
print(f"Minimum               : {generated_min:.8f}")
print(f"Maximum               : {generated_max:.8f}")
print(f"Unique vectors (6dp)  : {unique_generated}/{len(generated_np)}")

print()
print("FROZEN BASELINE SEMANTIC OUTPUT")
print("-" * 70)
print(f"Logit mean            : {logit_mean:.8f}")
print(f"Logit std             : {logit_std:.8f}")
print(f"Logit minimum         : {logit_min:.8f}")
print(f"Logit maximum         : {logit_max:.8f}")

print()
print("SEMANTIC PROBABILITIES")
print("-" * 70)
print(f"Mean                  : {probability_mean:.8f}")
print(f"Std                   : {probability_std:.8f}")
print(f"Minimum               : {probability_min:.8f}")
print(f"Maximum               : {probability_max:.8f}")

print()
print("INDEPENDENT ROC-AUC CHECK")
print("-" * 70)
print(f"Diagnostic ROC-AUC    : {diagnostic_auc:.10f}")

print()
print("=" * 100)
print("03.06 GAN TRAINING — DIAGNOSTIC COMPLETE")
print("=" * 100)

03.06 GAN TRAINING — POST-TRAINING DIAGNOSTIC

VALIDATION LABEL DISTRIBUTION
----------------------------------------------------------------------
Label 0 : 224 samples
Label 1 : 224 samples

REAL TARGET REPRESENTATIONS
----------------------------------------------------------------------
Mean                  : 0.50291955
Std                   : 2.22309780
Minimum               : -6.36920404
Maximum               : 8.91557980

GENERATED TARGET REPRESENTATIONS
----------------------------------------------------------------------
Mean                  : 0.11246932
Std                   : 1.38182640
Minimum               : -2.74368501
Maximum               : 3.31507277
Unique vectors (6dp)  : 200/448

FROZEN BASELINE SEMANTIC OUTPUT
----------------------------------------------------------------------
Logit mean            : -0.34556144
Logit std             : 0.58183599
Logit minimum         : -2.20136070
Logit maximum         : 0.45433241

SEMANTIC PROBABILITIES
-------------------

In [25]:
# ==================================================================================================
# CELL 24 — FROZEN BASELINE SANITY CHECK ON REAL VALIDATION TARGETS
# ==================================================================================================

import numpy as np
import torch
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

print("=" * 100)
print("03.06 GAN TRAINING — FROZEN BASELINE SANITY CHECK")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# Baseline must remain frozen
# --------------------------------------------------------------------------------------------------

baseline_model.eval()

for parameter in baseline_model.parameters():
    parameter.requires_grad = False

# --------------------------------------------------------------------------------------------------
# Collect predictions using REAL validation targets
# --------------------------------------------------------------------------------------------------

real_target_labels = []
real_target_logits = []
real_target_probabilities = []
real_target_predictions = []

with torch.no_grad():

    for source, real_target, labels in validation_loader:

        source = source.to(device)
        real_target = real_target.to(device)
        labels = labels.to(device)

        # ------------------------------------------------------------------
        # Frozen baseline receives the ORIGINAL target representation
        # ------------------------------------------------------------------

        semantic_logits = baseline_model(
            source,
            real_target
        )

        probabilities = torch.sigmoid(
            semantic_logits
        )

        predictions = (
            probabilities >= 0.5
        ).long()

        # ------------------------------------------------------------------
        # Store results
        # ------------------------------------------------------------------

        real_target_labels.extend(
            labels.cpu().numpy().tolist()
        )

        real_target_logits.extend(
            semantic_logits.cpu().numpy().tolist()
        )

        real_target_probabilities.extend(
            probabilities.cpu().numpy().tolist()
        )

        real_target_predictions.extend(
            predictions.cpu().numpy().tolist()
        )

# --------------------------------------------------------------------------------------------------
# Convert to NumPy
# --------------------------------------------------------------------------------------------------

real_target_labels = np.asarray(
    real_target_labels
)

real_target_logits = np.asarray(
    real_target_logits
)

real_target_probabilities = np.asarray(
    real_target_probabilities
)

real_target_predictions = np.asarray(
    real_target_predictions
)

# --------------------------------------------------------------------------------------------------
# Calculate metrics
# --------------------------------------------------------------------------------------------------

baseline_real_auc = roc_auc_score(
    real_target_labels,
    real_target_probabilities
)

baseline_real_accuracy = accuracy_score(
    real_target_labels,
    real_target_predictions
)

baseline_real_balanced_accuracy = balanced_accuracy_score(
    real_target_labels,
    real_target_predictions
)

baseline_real_precision = precision_score(
    real_target_labels,
    real_target_predictions,
    zero_division=0
)

baseline_real_recall = recall_score(
    real_target_labels,
    real_target_predictions,
    zero_division=0
)

baseline_real_f1 = f1_score(
    real_target_labels,
    real_target_predictions,
    zero_division=0
)

# --------------------------------------------------------------------------------------------------
# Print results
# --------------------------------------------------------------------------------------------------

print()
print("BASELINE PERFORMANCE USING REAL VALIDATION TARGETS")
print("-" * 70)

print(f"ROC-AUC               : {baseline_real_auc:.10f}")
print(f"Accuracy              : {baseline_real_accuracy:.10f}")
print(f"Balanced Accuracy     : {baseline_real_balanced_accuracy:.10f}")
print(f"Precision             : {baseline_real_precision:.10f}")
print(f"Recall                : {baseline_real_recall:.10f}")
print(f"F1 Score              : {baseline_real_f1:.10f}")

print()
print("BASELINE OUTPUT DISTRIBUTION")
print("-" * 70)

print(
    f"Logit mean            : "
    f"{real_target_logits.mean():.8f}"
)

print(
    f"Logit std             : "
    f"{real_target_logits.std():.8f}"
)

print(
    f"Probability mean      : "
    f"{real_target_probabilities.mean():.8f}"
)

print(
    f"Probability std       : "
    f"{real_target_probabilities.std():.8f}"
)

print(
    f"Probability minimum   : "
    f"{real_target_probabilities.min():.8f}"
)

print(
    f"Probability maximum   : "
    f"{real_target_probabilities.max():.8f}"
)

# --------------------------------------------------------------------------------------------------
# Compare against official baseline result
# --------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("COMPARISON WITH OFFICIAL 03.04 BASELINE")
print("=" * 100)

print(
    f"Official 03.04 ROC-AUC : "
    f"{BASELINE_REFERENCE_ROC_AUC:.10f}"
)

print(
    f"Current checkpoint AUC : "
    f"{baseline_real_auc:.10f}"
)

print(
    f"Absolute difference    : "
    f"{abs(baseline_real_auc - BASELINE_REFERENCE_ROC_AUC):.10f}"
)

if abs(
    baseline_real_auc - BASELINE_REFERENCE_ROC_AUC
) < 1e-6:

    print()
    print("✓ BASELINE SANITY CHECK PASSED")
    print("✓ Frozen baseline reproduces the official validation ROC-AUC")

else:

    print()
    print("⚠ BASELINE SANITY CHECK DIFFERENCE DETECTED")
    print("⚠ The current validation evaluation does not exactly reproduce")
    print("  the official 03.04 baseline ROC-AUC.")

print()
print("=" * 100)
print("FROZEN BASELINE SANITY CHECK COMPLETE")
print("=" * 100)

03.06 GAN TRAINING — FROZEN BASELINE SANITY CHECK

BASELINE PERFORMANCE USING REAL VALIDATION TARGETS
----------------------------------------------------------------------
ROC-AUC               : 0.6626674107
Accuracy              : 0.6116071429
Balanced Accuracy     : 0.6116071429
Precision             : 0.6068376068
Recall                : 0.6339285714
F1 Score              : 0.6200873362

BASELINE OUTPUT DISTRIBUTION
----------------------------------------------------------------------
Logit mean            : 0.07046689
Logit std             : 1.01212390
Probability mean      : 0.51538370
Probability std       : 0.21051548
Probability minimum   : 0.03269292
Probability maximum   : 0.93790358

COMPARISON WITH OFFICIAL 03.04 BASELINE
Official 03.04 ROC-AUC : 0.6626670000
Current checkpoint AUC : 0.6626674107
Absolute difference    : 0.0000004107

✓ BASELINE SANITY CHECK PASSED
✓ Frozen baseline reproduces the official validation ROC-AUC

FROZEN BASELINE SANITY CHECK COMPLETE


In [26]:
# ==================================================================================================
# CELL 25 — REAL vs GENERATED TARGET REPRESENTATION COMPARISON
# ==================================================================================================

import numpy as np
import torch

print("=" * 100)
print("03.06 GAN TRAINING — REAL vs GENERATED TARGET ANALYSIS")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# Best Generator is already loaded as diagnostic_generator
# Baseline remains frozen
# --------------------------------------------------------------------------------------------------

diagnostic_generator.eval()
baseline_model.eval()

for parameter in baseline_model.parameters():
    parameter.requires_grad = False

# --------------------------------------------------------------------------------------------------
# Collect real and generated validation representations
# --------------------------------------------------------------------------------------------------

real_targets = []
generated_targets = []
source_representations = []
labels_collection = []

with torch.no_grad():

    for source, real_target, labels in validation_loader:

        source = source.to(device)
        real_target = real_target.to(device)

        generated_target = diagnostic_generator(source)

        source_representations.append(
            source.cpu()
        )

        real_targets.append(
            real_target.cpu()
        )

        generated_targets.append(
            generated_target.cpu()
        )

        labels_collection.append(
            labels.cpu()
        )

# --------------------------------------------------------------------------------------------------
# Concatenate
# --------------------------------------------------------------------------------------------------

source_representations = torch.cat(
    source_representations,
    dim=0
)

real_targets = torch.cat(
    real_targets,
    dim=0
)

generated_targets = torch.cat(
    generated_targets,
    dim=0
)

labels_collection = torch.cat(
    labels_collection,
    dim=0
)

# --------------------------------------------------------------------------------------------------
# Convert to NumPy
# --------------------------------------------------------------------------------------------------

source_np = source_representations.numpy()
real_np = real_targets.numpy()
generated_np = generated_targets.numpy()
labels_np = labels_collection.numpy()

# --------------------------------------------------------------------------------------------------
# Global statistics
# --------------------------------------------------------------------------------------------------

real_mean = real_np.mean()
generated_mean = generated_np.mean()

real_std = real_np.std()
generated_std = generated_np.std()

mean_absolute_difference = np.mean(
    np.abs(real_np - generated_np)
)

mean_squared_difference = np.mean(
    (real_np - generated_np) ** 2
)

# --------------------------------------------------------------------------------------------------
# Per-dimension statistics
# --------------------------------------------------------------------------------------------------

real_dimension_means = real_np.mean(axis=0)
generated_dimension_means = generated_np.mean(axis=0)

real_dimension_stds = real_np.std(axis=0)
generated_dimension_stds = generated_np.std(axis=0)

mean_dimension_mean_difference = np.mean(
    np.abs(
        real_dimension_means -
        generated_dimension_means
    )
)

mean_dimension_std_difference = np.mean(
    np.abs(
        real_dimension_stds -
        generated_dimension_stds
    )
)

# --------------------------------------------------------------------------------------------------
# Norm statistics
# --------------------------------------------------------------------------------------------------

real_norms = np.linalg.norm(
    real_np,
    axis=1
)

generated_norms = np.linalg.norm(
    generated_np,
    axis=1
)

# --------------------------------------------------------------------------------------------------
# Print results
# --------------------------------------------------------------------------------------------------

print()
print("GLOBAL REPRESENTATION STATISTICS")
print("-" * 70)

print(
    f"Real target mean                  : "
    f"{real_mean:.8f}"
)

print(
    f"Generated target mean             : "
    f"{generated_mean:.8f}"
)

print(
    f"Real target std                   : "
    f"{real_std:.8f}"
)

print(
    f"Generated target std              : "
    f"{generated_std:.8f}"
)

print()
print("REAL vs GENERATED DIFFERENCE")
print("-" * 70)

print(
    f"Mean absolute difference           : "
    f"{mean_absolute_difference:.8f}"
)

print(
    f"Mean squared difference            : "
    f"{mean_squared_difference:.8f}"
)

print()
print("PER-DIMENSION STATISTICS")
print("-" * 70)

print(
    f"Mean absolute difference of means : "
    f"{mean_dimension_mean_difference:.8f}"
)

print(
    f"Mean absolute difference of stds   : "
    f"{mean_dimension_std_difference:.8f}"
)

print()
print("REPRESENTATION NORMS")
print("-" * 70)

print(
    f"Real norm mean                     : "
    f"{real_norms.mean():.8f}"
)

print(
    f"Generated norm mean                : "
    f"{generated_norms.mean():.8f}"
)

print(
    f"Real norm std                      : "
    f"{real_norms.std():.8f}"
)

print(
    f"Generated norm std                 : "
    f"{generated_norms.std():.8f}"
)

print(
    f"Real norm minimum                  : "
    f"{real_norms.min():.8f}"
)

print(
    f"Generated norm minimum             : "
    f"{generated_norms.min():.8f}"
)

print(
    f"Real norm maximum                  : "
    f"{real_norms.max():.8f}"
)

print(
    f"Generated norm maximum             : "
    f"{generated_norms.max():.8f}"
)

print()
print("=" * 100)
print("REAL vs GENERATED TARGET ANALYSIS COMPLETE")
print("=" * 100)

03.06 GAN TRAINING — REAL vs GENERATED TARGET ANALYSIS

GLOBAL REPRESENTATION STATISTICS
----------------------------------------------------------------------
Real target mean                  : 0.50291955
Generated target mean             : 0.11246933
Real target std                   : 2.22302032
Generated target std              : 1.38177812

REAL vs GENERATED DIFFERENCE
----------------------------------------------------------------------
Mean absolute difference           : 1.48924422
Mean squared difference            : 3.45625043

PER-DIMENSION STATISTICS
----------------------------------------------------------------------
Mean absolute difference of means : 1.22046304
Mean absolute difference of stds   : 0.85635984

REPRESENTATION NORMS
----------------------------------------------------------------------
Real norm mean                     : 12.18399239
Generated norm mean                : 7.77655506
Real norm std                      : 4.21690226
Generated norm std       

In [27]:
# ==================================================================================================
# CELL 26 — GAN TRAINING CURVE DIAGNOSTIC
# ==================================================================================================

import numpy as np

print("=" * 100)
print("03.06 GAN TRAINING — TRAINING CURVE DIAGNOSTIC")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# Convert training history to NumPy arrays
# --------------------------------------------------------------------------------------------------

epochs = np.asarray(
    training_history["epoch"]
)

generator_total = np.asarray(
    training_history["generator_total_loss"]
)

generator_adversarial = np.asarray(
    training_history["generator_adversarial_loss"]
)

generator_semantic = np.asarray(
    training_history["generator_semantic_loss"]
)

discriminator_total = np.asarray(
    training_history["discriminator_total_loss"]
)

discriminator_real = np.asarray(
    training_history["discriminator_real_loss"]
)

discriminator_fake = np.asarray(
    training_history["discriminator_fake_loss"]
)

validation_loss = np.asarray(
    training_history["validation_loss"]
)

validation_auc = np.asarray(
    training_history["validation_roc_auc"]
)

# --------------------------------------------------------------------------------------------------
# Best semantic-loss epoch
# --------------------------------------------------------------------------------------------------

best_semantic_loss_index = np.argmin(
    generator_semantic
)

best_semantic_loss_epoch = int(
    epochs[best_semantic_loss_index]
)

best_semantic_loss_value = float(
    generator_semantic[best_semantic_loss_index]
)

# --------------------------------------------------------------------------------------------------
# Best validation AUC epoch
# --------------------------------------------------------------------------------------------------

best_auc_index = np.argmax(
    validation_auc
)

best_auc_epoch = int(
    epochs[best_auc_index]
)

best_auc_value = float(
    validation_auc[best_auc_index]
)

# --------------------------------------------------------------------------------------------------
# Initial and final values
# --------------------------------------------------------------------------------------------------

print()
print("GENERATOR LOSS")
print("-" * 70)

print(
    f"Initial total loss       : "
    f"{generator_total[0]:.8f}"
)

print(
    f"Final total loss         : "
    f"{generator_total[-1]:.8f}"
)

print(
    f"Initial adversarial loss : "
    f"{generator_adversarial[0]:.8f}"
)

print(
    f"Final adversarial loss   : "
    f"{generator_adversarial[-1]:.8f}"
)

print(
    f"Initial semantic loss    : "
    f"{generator_semantic[0]:.8f}"
)

print(
    f"Final semantic loss      : "
    f"{generator_semantic[-1]:.8f}"
)

print()
print("BEST GENERATOR SEMANTIC LOSS")
print("-" * 70)

print(
    f"Best semantic-loss epoch : "
    f"{best_semantic_loss_epoch}"
)

print(
    f"Best semantic loss       : "
    f"{best_semantic_loss_value:.8f}"
)

print()
print("DISCRIMINATOR LOSS")
print("-" * 70)

print(
    f"Initial total loss       : "
    f"{discriminator_total[0]:.8f}"
)

print(
    f"Final total loss         : "
    f"{discriminator_total[-1]:.8f}"
)

print(
    f"Initial real loss        : "
    f"{discriminator_real[0]:.8f}"
)

print(
    f"Final real loss          : "
    f"{discriminator_real[-1]:.8f}"
)

print(
    f"Initial fake loss        : "
    f"{discriminator_fake[0]:.8f}"
)

print(
    f"Final fake loss          : "
    f"{discriminator_fake[-1]:.8f}"
)

print()
print("VALIDATION")
print("-" * 70)

print(
    f"Initial validation loss  : "
    f"{validation_loss[0]:.8f}"
)

print(
    f"Final validation loss    : "
    f"{validation_loss[-1]:.8f}"
)

print(
    f"Best validation AUC      : "
    f"{best_auc_value:.10f}"
)

print(
    f"Best AUC epoch           : "
    f"{best_auc_epoch}"
)

print()
print("FIRST 10 EPOCHS — SEMANTIC LOSS vs VALIDATION AUC")
print("-" * 70)

for index in range(min(10, len(epochs))):

    print(
        f"Epoch {int(epochs[index]):02d} | "
        f"Semantic Loss: {generator_semantic[index]:.6f} | "
        f"Val AUC: {validation_auc[index]:.10f}"
    )

print()
print("LAST 10 EPOCHS — SEMANTIC LOSS vs VALIDATION AUC")
print("-" * 70)

start_index = max(0, len(epochs) - 10)

for index in range(start_index, len(epochs)):

    print(
        f"Epoch {int(epochs[index]):02d} | "
        f"Semantic Loss: {generator_semantic[index]:.6f} | "
        f"Val AUC: {validation_auc[index]:.10f}"
    )

print()
print("=" * 100)
print("TRAINING CURVE DIAGNOSTIC COMPLETE")
print("=" * 100)

03.06 GAN TRAINING — TRAINING CURVE DIAGNOSTIC

GENERATOR LOSS
----------------------------------------------------------------------
Initial total loss       : 1.39120084
Final total loss         : 1.31651016
Initial adversarial loss : 0.68183537
Final adversarial loss   : 0.61750213
Initial semantic loss    : 0.70936548
Final semantic loss      : 0.69900802

BEST GENERATOR SEMANTIC LOSS
----------------------------------------------------------------------
Best semantic-loss epoch : 49
Best semantic loss       : 0.69679009

DISCRIMINATOR LOSS
----------------------------------------------------------------------
Initial total loss       : 0.86064333
Final total loss         : 1.48464940
Initial real loss        : 0.14940429
Final real loss          : 0.59418985
Initial fake loss        : 0.71123904
Final fake loss          : 0.89045955

VALIDATION
----------------------------------------------------------------------
Initial validation loss  : 0.69698319
Final validation loss    : 0.

In [28]:
# ==================================================================================================
# CELL 27 — SEMANTIC-ONLY GENERATOR ABLATION DIAGNOSTIC
# ==================================================================================================

import numpy as np
import torch

print("=" * 100)
print("03.06 GAN TRAINING — SEMANTIC-ONLY GENERATOR DIAGNOSTIC")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# Create a fresh temporary Generator
# --------------------------------------------------------------------------------------------------

torch.manual_seed(SEED)

semantic_test_generator = Generator().to(device)

semantic_test_generator.train()

# --------------------------------------------------------------------------------------------------
# Baseline remains frozen
# --------------------------------------------------------------------------------------------------

baseline_model.eval()

for parameter in baseline_model.parameters():
    parameter.requires_grad = False

# --------------------------------------------------------------------------------------------------
# Temporary optimizer
# --------------------------------------------------------------------------------------------------

semantic_test_optimizer = torch.optim.Adam(
    semantic_test_generator.parameters(),
    lr=GAN_LEARNING_RATE
)

# --------------------------------------------------------------------------------------------------
# Fixed diagnostic batch
# --------------------------------------------------------------------------------------------------

diagnostic_batch = next(
    iter(validation_loader)
)

diagnostic_source = diagnostic_batch[0].to(device)
diagnostic_labels = diagnostic_batch[2].to(device)

print()
print("DIAGNOSTIC BATCH")
print("-" * 70)
print(f"Source shape            : {tuple(diagnostic_source.shape)}")
print(f"Labels shape            : {tuple(diagnostic_labels.shape)}")
print(
    f"Positive labels         : "
    f"{int((diagnostic_labels == 1).sum().item())}"
)
print(
    f"Negative labels         : "
    f"{int((diagnostic_labels == 0).sum().item())}"
)

# --------------------------------------------------------------------------------------------------
# Measure semantic loss BEFORE optimization
# --------------------------------------------------------------------------------------------------

with torch.no_grad():

    initial_generated = semantic_test_generator(
        diagnostic_source
    )

    initial_logits = baseline_model(
        diagnostic_source,
        initial_generated
    )

    initial_loss = semantic_loss_fn(
        initial_logits,
        diagnostic_labels
    )

    initial_probabilities = torch.sigmoid(
        initial_logits
    )

initial_loss_value = initial_loss.item()

# --------------------------------------------------------------------------------------------------
# Semantic-only optimization
# --------------------------------------------------------------------------------------------------

NUM_DIAGNOSTIC_STEPS = 100

semantic_losses = []

for step in range(1, NUM_DIAGNOSTIC_STEPS + 1):

    semantic_test_optimizer.zero_grad()

    generated_target = semantic_test_generator(
        diagnostic_source
    )

    semantic_logits = baseline_model(
        diagnostic_source,
        generated_target
    )

    semantic_loss = semantic_loss_fn(
        semantic_logits,
        diagnostic_labels
    )

    semantic_loss.backward()

    semantic_test_optimizer.step()

    semantic_losses.append(
        semantic_loss.item()
    )

# --------------------------------------------------------------------------------------------------
# Measure semantic loss AFTER optimization
# --------------------------------------------------------------------------------------------------

semantic_test_generator.eval()

with torch.no_grad():

    final_generated = semantic_test_generator(
        diagnostic_source
    )

    final_logits = baseline_model(
        diagnostic_source,
        final_generated
    )

    final_loss = semantic_loss_fn(
        final_logits,
        diagnostic_labels
    )

    final_probabilities = torch.sigmoid(
        final_logits
    )

final_loss_value = final_loss.item()

# --------------------------------------------------------------------------------------------------
# Calculate probability statistics
# --------------------------------------------------------------------------------------------------

initial_probability_std = (
    initial_probabilities.std().item()
)

final_probability_std = (
    final_probabilities.std().item()
)

initial_probability_mean = (
    initial_probabilities.mean().item()
)

final_probability_mean = (
    final_probabilities.mean().item()
)

# --------------------------------------------------------------------------------------------------
# Print results
# --------------------------------------------------------------------------------------------------

print()
print("SEMANTIC-ONLY OPTIMIZATION")
print("-" * 70)

print(
    f"Optimization steps       : "
    f"{NUM_DIAGNOSTIC_STEPS}"
)

print(
    f"Initial semantic loss    : "
    f"{initial_loss_value:.10f}"
)

print(
    f"Final semantic loss      : "
    f"{final_loss_value:.10f}"
)

print(
    f"Loss change              : "
    f"{final_loss_value - initial_loss_value:.10f}"
)

print()
print("SEMANTIC PROBABILITY DISTRIBUTION")
print("-" * 70)

print(
    f"Initial probability mean : "
    f"{initial_probability_mean:.8f}"
)

print(
    f"Final probability mean   : "
    f"{final_probability_mean:.8f}"
)

print(
    f"Initial probability std  : "
    f"{initial_probability_std:.8f}"
)

print(
    f"Final probability std    : "
    f"{final_probability_std:.8f}"
)

print()
print("=" * 100)

if final_loss_value < initial_loss_value:

    print("✓ SEMANTIC LOSS DECREASED")
    print("✓ Frozen baseline provides a usable learning signal")
    print("→ Adversarial training may be interfering with semantic preservation")

else:

    print("⚠ SEMANTIC LOSS DID NOT DECREASE")
    print("⚠ The semantic objective itself may be insufficient")
    print("→ Further investigation of the semantic objective is required")

print("=" * 100)
print("SEMANTIC-ONLY DIAGNOSTIC COMPLETE")
print("=" * 100)

03.06 GAN TRAINING — SEMANTIC-ONLY GENERATOR DIAGNOSTIC

DIAGNOSTIC BATCH
----------------------------------------------------------------------
Source shape            : (32, 32)
Labels shape            : (32,)
Positive labels         : 24
Negative labels         : 8

SEMANTIC-ONLY OPTIMIZATION
----------------------------------------------------------------------
Optimization steps       : 100
Initial semantic loss    : 0.8221482635
Final semantic loss      : 0.3470866680
Loss change              : -0.4750615954

SEMANTIC PROBABILITY DISTRIBUTION
----------------------------------------------------------------------
Initial probability mean : 0.45803088
Final probability mean   : 0.75342810
Initial probability std  : 0.10316630
Final probability std    : 0.25021636

✓ SEMANTIC LOSS DECREASED
✓ Frozen baseline provides a usable learning signal
→ Adversarial training may be interfering with semantic preservation
SEMANTIC-ONLY DIAGNOSTIC COMPLETE


In [29]:
# ==================================================================================================
# CELL 28 — ADVERSARIAL vs SEMANTIC GRADIENT DIAGNOSTIC
# ==================================================================================================

import numpy as np
import torch

print("=" * 100)
print("03.06 GAN TRAINING — ADVERSARIAL vs SEMANTIC GRADIENT ANALYSIS")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# Create fresh temporary models
# --------------------------------------------------------------------------------------------------

torch.manual_seed(SEED)

gradient_test_generator = Generator().to(device)
gradient_test_discriminator = Discriminator().to(device)

gradient_test_generator.train()
gradient_test_discriminator.train()

baseline_model.eval()

for parameter in baseline_model.parameters():
    parameter.requires_grad = False

# --------------------------------------------------------------------------------------------------
# Get one fixed training batch
# --------------------------------------------------------------------------------------------------

gradient_batch = next(
    iter(train_loader)
)

gradient_source = gradient_batch[0].to(device)
gradient_real_target = gradient_batch[1].to(device)
gradient_labels = gradient_batch[2].to(device)

print()
print("DIAGNOSTIC BATCH")
print("-" * 70)
print(f"Source shape            : {tuple(gradient_source.shape)}")
print(f"Real target shape       : {tuple(gradient_real_target.shape)}")
print(f"Labels shape            : {tuple(gradient_labels.shape)}")

# --------------------------------------------------------------------------------------------------
# Helper function for total gradient norm
# --------------------------------------------------------------------------------------------------

def calculate_gradient_norm(model):

    total_squared_norm = 0.0

    for parameter in model.parameters():

        if parameter.grad is not None:

            parameter_gradient_norm = (
                parameter.grad.detach().norm(2).item()
            )

            total_squared_norm += (
                parameter_gradient_norm ** 2
            )

    return total_squared_norm ** 0.5

# ==================================================================================================
# 1. ADVERSARIAL GRADIENT
# ==================================================================================================

gradient_test_generator.zero_grad()
gradient_test_discriminator.zero_grad()

generated_target = gradient_test_generator(
    gradient_source
)

fake_logits = gradient_test_discriminator(
    generated_target
)

generator_real_labels = torch.ones_like(
    fake_logits
)

adversarial_loss = adversarial_loss_fn(
    fake_logits,
    generator_real_labels
)

adversarial_loss.backward()

adversarial_gradient_norm = calculate_gradient_norm(
    gradient_test_generator
)

# ==================================================================================================
# 2. SEMANTIC GRADIENT
# ==================================================================================================

gradient_test_generator.zero_grad()
gradient_test_discriminator.zero_grad()

generated_target = gradient_test_generator(
    gradient_source
)

semantic_logits = baseline_model(
    gradient_source,
    generated_target
)

semantic_loss = semantic_loss_fn(
    semantic_logits,
    gradient_labels
)

semantic_loss.backward()

semantic_gradient_norm = calculate_gradient_norm(
    gradient_test_generator
)

# ==================================================================================================
# 3. COMBINED GRADIENT
# ==================================================================================================

gradient_test_generator.zero_grad()
gradient_test_discriminator.zero_grad()

generated_target = gradient_test_generator(
    gradient_source
)

fake_logits = gradient_test_discriminator(
    generated_target
)

generator_real_labels = torch.ones_like(
    fake_logits
)

combined_adversarial_loss = adversarial_loss_fn(
    fake_logits,
    generator_real_labels
)

semantic_logits = baseline_model(
    gradient_source,
    generated_target
)

combined_semantic_loss = semantic_loss_fn(
    semantic_logits,
    gradient_labels
)

combined_loss = (
    combined_adversarial_loss +
    SEMANTIC_LOSS_WEIGHT * combined_semantic_loss
)

combined_loss.backward()

combined_gradient_norm = calculate_gradient_norm(
    gradient_test_generator
)

# ==================================================================================================
# Print results
# ==================================================================================================

print()
print("LOSS VALUES")
print("-" * 70)

print(
    f"Adversarial loss        : "
    f"{adversarial_loss.item():.10f}"
)

print(
    f"Semantic loss           : "
    f"{semantic_loss.item():.10f}"
)

print(
    f"Combined loss           : "
    f"{combined_loss.item():.10f}"
)

print()
print("GENERATOR GRADIENT NORMS")
print("-" * 70)

print(
    f"Adversarial gradient    : "
    f"{adversarial_gradient_norm:.10f}"
)

print(
    f"Semantic gradient       : "
    f"{semantic_gradient_norm:.10f}"
)

print(
    f"Combined gradient       : "
    f"{combined_gradient_norm:.10f}"
)

# --------------------------------------------------------------------------------------------------
# Relative gradient strength
# --------------------------------------------------------------------------------------------------

if semantic_gradient_norm > 0:

    adversarial_to_semantic_ratio = (
        adversarial_gradient_norm /
        semantic_gradient_norm
    )

else:

    adversarial_to_semantic_ratio = np.inf

print()
print("RELATIVE GRADIENT STRENGTH")
print("-" * 70)

print(
    f"Adversarial / Semantic  : "
    f"{adversarial_to_semantic_ratio:.10f}"
)

print()
print("=" * 100)
print("GRADIENT DIAGNOSTIC COMPLETE")
print("=" * 100)

03.06 GAN TRAINING — ADVERSARIAL vs SEMANTIC GRADIENT ANALYSIS

DIAGNOSTIC BATCH
----------------------------------------------------------------------
Source shape            : (32, 32)
Real target shape       : (32, 32)
Labels shape            : (32,)

LOSS VALUES
----------------------------------------------------------------------
Adversarial loss        : 0.6155654192
Semantic loss           : 0.6648020744
Combined loss           : 1.2803674936

GENERATOR GRADIENT NORMS
----------------------------------------------------------------------
Adversarial gradient    : 0.0567732887
Semantic gradient       : 0.3006858220
Combined gradient       : 0.3121239522

RELATIVE GRADIENT STRENGTH
----------------------------------------------------------------------
Adversarial / Semantic  : 0.1888126562

GRADIENT DIAGNOSTIC COMPLETE


In [30]:
# ==================================================================================================
# CELL 29 — GENERATOR GRADIENT DIRECTION CONFLICT TEST
# ==================================================================================================

import torch
import torch.nn.functional as F

print("=" * 100)
print("03.06 GAN TRAINING — GRADIENT DIRECTION CONFLICT TEST")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# Fresh temporary models
# --------------------------------------------------------------------------------------------------

torch.manual_seed(SEED)

direction_generator = Generator().to(device)
direction_discriminator = Discriminator().to(device)

direction_generator.train()
direction_discriminator.train()

baseline_model.eval()

for parameter in baseline_model.parameters():
    parameter.requires_grad = False

# --------------------------------------------------------------------------------------------------
# Fixed diagnostic batch
# --------------------------------------------------------------------------------------------------

direction_batch = next(
    iter(train_loader)
)

direction_source = direction_batch[0].to(device)
direction_labels = direction_batch[2].to(device)

print()
print("DIAGNOSTIC BATCH")
print("-" * 70)
print(f"Source shape            : {tuple(direction_source.shape)}")
print(f"Labels shape            : {tuple(direction_labels.shape)}")

# --------------------------------------------------------------------------------------------------
# Helper: flatten Generator gradients
# --------------------------------------------------------------------------------------------------

def get_flattened_generator_gradient(model):

    gradients = []

    for parameter in model.parameters():

        if parameter.grad is not None:
            gradients.append(
                parameter.grad.detach().flatten()
            )

    return torch.cat(gradients)

# ==================================================================================================
# 1. ADVERSARIAL GRADIENT
# ==================================================================================================

direction_generator.zero_grad()
direction_discriminator.zero_grad()

generated_target = direction_generator(
    direction_source
)

fake_logits = direction_discriminator(
    generated_target
)

generator_real_labels = torch.ones_like(
    fake_logits
)

adversarial_loss = adversarial_loss_fn(
    fake_logits,
    generator_real_labels
)

adversarial_loss.backward()

adversarial_gradient_vector = (
    get_flattened_generator_gradient(
        direction_generator
    )
)

# ==================================================================================================
# 2. SEMANTIC GRADIENT
# ==================================================================================================

direction_generator.zero_grad()
direction_discriminator.zero_grad()

generated_target = direction_generator(
    direction_source
)

semantic_logits = baseline_model(
    direction_source,
    generated_target
)

semantic_loss = semantic_loss_fn(
    semantic_logits,
    direction_labels
)

semantic_loss.backward()

semantic_gradient_vector = (
    get_flattened_generator_gradient(
        direction_generator
    )
)

# ==================================================================================================
# 3. GRADIENT MAGNITUDES
# ==================================================================================================

adversarial_gradient_norm = (
    torch.norm(
        adversarial_gradient_vector,
        p=2
    ).item()
)

semantic_gradient_norm = (
    torch.norm(
        semantic_gradient_vector,
        p=2
    ).item()
)

# ==================================================================================================
# 4. COSINE SIMILARITY
# ==================================================================================================

gradient_cosine_similarity = F.cosine_similarity(
    adversarial_gradient_vector.unsqueeze(0),
    semantic_gradient_vector.unsqueeze(0),
    dim=1
).item()

# ==================================================================================================
# 5. ANGLE BETWEEN GRADIENTS
# ==================================================================================================

gradient_cosine_clipped = np.clip(
    gradient_cosine_similarity,
    -1.0,
    1.0
)

gradient_angle_degrees = (
    np.degrees(
        np.arccos(
            gradient_cosine_clipped
        )
    )
)

# ==================================================================================================
# Print results
# ==================================================================================================

print()
print("GRADIENT MAGNITUDES")
print("-" * 70)

print(
    f"Adversarial gradient norm : "
    f"{adversarial_gradient_norm:.10f}"
)

print(
    f"Semantic gradient norm    : "
    f"{semantic_gradient_norm:.10f}"
)

print()
print("GRADIENT DIRECTION")
print("-" * 70)

print(
    f"Cosine similarity         : "
    f"{gradient_cosine_similarity:.10f}"
)

print(
    f"Gradient angle            : "
    f"{gradient_angle_degrees:.4f} degrees"
)

print()
print("=" * 100)

if gradient_cosine_similarity < 0:

    print("⚠ GRADIENT CONFLICT DETECTED")
    print("⚠ Adversarial and semantic objectives point in opposing directions")

elif gradient_cosine_similarity < 0.25:

    print("⚠ WEAK GRADIENT ALIGNMENT")
    print("⚠ The two objectives are largely directionally incompatible")

elif gradient_cosine_similarity < 0.75:

    print("✓ MODERATE GRADIENT ALIGNMENT")
    print("→ Objectives partially cooperate but are not strongly aligned")

else:

    print("✓ STRONG GRADIENT ALIGNMENT")
    print("→ Objectives largely point in the same direction")

print("=" * 100)
print("GRADIENT DIRECTION TEST COMPLETE")
print("=" * 100)

03.06 GAN TRAINING — GRADIENT DIRECTION CONFLICT TEST

DIAGNOSTIC BATCH
----------------------------------------------------------------------
Source shape            : (32, 32)
Labels shape            : (32,)

GRADIENT MAGNITUDES
----------------------------------------------------------------------
Adversarial gradient norm : 0.0570977665
Semantic gradient norm    : 0.7501736283

GRADIENT DIRECTION
----------------------------------------------------------------------
Cosine similarity         : 0.1136652976
Gradient angle            : 83.4734 degrees

⚠ WEAK GRADIENT ALIGNMENT
⚠ The two objectives are largely directionally incompatible
GRADIENT DIRECTION TEST COMPLETE


In [32]:
# ==================================================================================================
# CELL 29 — GENERATOR GRADIENT DIRECTION CONFLICT TEST
# ==================================================================================================

import torch
import torch.nn.functional as F

print("=" * 100)
print("03.06 GAN TRAINING — GRADIENT DIRECTION CONFLICT TEST")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# Fresh temporary models
# --------------------------------------------------------------------------------------------------

torch.manual_seed(SEED)

direction_generator = Generator().to(device)
direction_discriminator = Discriminator().to(device)

direction_generator.train()
direction_discriminator.train()

baseline_model.eval()

for parameter in baseline_model.parameters():
    parameter.requires_grad = False

# --------------------------------------------------------------------------------------------------
# Fixed diagnostic batch
# --------------------------------------------------------------------------------------------------

direction_batch = next(
    iter(train_loader)
)

direction_source = direction_batch[0].to(device)
direction_labels = direction_batch[2].to(device)

print()
print("DIAGNOSTIC BATCH")
print("-" * 70)
print(f"Source shape            : {tuple(direction_source.shape)}")
print(f"Labels shape            : {tuple(direction_labels.shape)}")

# --------------------------------------------------------------------------------------------------
# Helper: flatten Generator gradients
# --------------------------------------------------------------------------------------------------

def get_flattened_generator_gradient(model):

    gradients = []

    for parameter in model.parameters():

        if parameter.grad is not None:
            gradients.append(
                parameter.grad.detach().flatten()
            )

    return torch.cat(gradients)

# ==================================================================================================
# 1. ADVERSARIAL GRADIENT
# ==================================================================================================

direction_generator.zero_grad()
direction_discriminator.zero_grad()

generated_target = direction_generator(
    direction_source
)

fake_logits = direction_discriminator(
    generated_target
)

generator_real_labels = torch.ones_like(
    fake_logits
)

adversarial_loss = adversarial_loss_fn(
    fake_logits,
    generator_real_labels
)

adversarial_loss.backward()

adversarial_gradient_vector = (
    get_flattened_generator_gradient(
        direction_generator
    )
)

# ==================================================================================================
# 2. SEMANTIC GRADIENT
# ==================================================================================================

direction_generator.zero_grad()
direction_discriminator.zero_grad()

generated_target = direction_generator(
    direction_source
)

semantic_logits = baseline_model(
    direction_source,
    generated_target
)

semantic_loss = semantic_loss_fn(
    semantic_logits,
    direction_labels
)

semantic_loss.backward()

semantic_gradient_vector = (
    get_flattened_generator_gradient(
        direction_generator
    )
)

# ==================================================================================================
# 3. GRADIENT MAGNITUDES
# ==================================================================================================

adversarial_gradient_norm = (
    torch.norm(
        adversarial_gradient_vector,
        p=2
    ).item()
)

semantic_gradient_norm = (
    torch.norm(
        semantic_gradient_vector,
        p=2
    ).item()
)

# ==================================================================================================
# 4. COSINE SIMILARITY
# ==================================================================================================

gradient_cosine_similarity = F.cosine_similarity(
    adversarial_gradient_vector.unsqueeze(0),
    semantic_gradient_vector.unsqueeze(0),
    dim=1
).item()

# ==================================================================================================
# 5. ANGLE BETWEEN GRADIENTS
# ==================================================================================================

gradient_cosine_clipped = np.clip(
    gradient_cosine_similarity,
    -1.0,
    1.0
)

gradient_angle_degrees = (
    np.degrees(
        np.arccos(
            gradient_cosine_clipped
        )
    )
)

# ==================================================================================================
# Print results
# ==================================================================================================

print()
print("GRADIENT MAGNITUDES")
print("-" * 70)

print(
    f"Adversarial gradient norm : "
    f"{adversarial_gradient_norm:.10f}"
)

print(
    f"Semantic gradient norm    : "
    f"{semantic_gradient_norm:.10f}"
)

print()
print("GRADIENT DIRECTION")
print("-" * 70)

print(
    f"Cosine similarity         : "
    f"{gradient_cosine_similarity:.10f}"
)

print(
    f"Gradient angle            : "
    f"{gradient_angle_degrees:.4f} degrees"
)

print()
print("=" * 100)

if gradient_cosine_similarity < 0:

    print("⚠ GRADIENT CONFLICT DETECTED")
    print("⚠ Adversarial and semantic objectives point in opposing directions")

elif gradient_cosine_similarity < 0.25:

    print("⚠ WEAK GRADIENT ALIGNMENT")
    print("⚠ The two objectives are largely directionally incompatible")

elif gradient_cosine_similarity < 0.75:

    print("✓ MODERATE GRADIENT ALIGNMENT")
    print("→ Objectives partially cooperate but are not strongly aligned")

else:

    print("✓ STRONG GRADIENT ALIGNMENT")
    print("→ Objectives largely point in the same direction")

print("=" * 100)
print("GRADIENT DIRECTION TEST COMPLETE")
print("=" * 100)

03.06 GAN TRAINING — GRADIENT DIRECTION CONFLICT TEST

DIAGNOSTIC BATCH
----------------------------------------------------------------------
Source shape            : (32, 32)
Labels shape            : (32,)

GRADIENT MAGNITUDES
----------------------------------------------------------------------
Adversarial gradient norm : 0.0568154864
Semantic gradient norm    : 0.4703998268

GRADIENT DIRECTION
----------------------------------------------------------------------
Cosine similarity         : 0.1185325235
Gradient angle            : 83.1926 degrees

⚠ WEAK GRADIENT ALIGNMENT
⚠ The two objectives are largely directionally incompatible
GRADIENT DIRECTION TEST COMPLETE
